# FINAL: Sixway + H6 SWA EMA14-16 (Core6, VM), per-target percentile-rank blend
Same cells as p3_swa_mix (57519642); stage 2 with the deadline rule (studies H6 cannot reach take the Sixway rank). Any missing stage output leaves no submission.csv.


In [ ]:
# Stage 1: Sixway resident cell (359192097) byte for byte, run as an isolated child process.
import base64, hashlib, io, json, os, shutil, subprocess, sys, time
from pathlib import Path
import numpy as np, pandas as pd
WORK = Path('/kaggle/working')
TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
RD = dict(dtype={'StudyInstanceUID': str}, float_precision='round_trip')
def file_sha256(p):
    with Path(p).open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()
def valid_frame(frame, uids=None):
    v = frame[TARGETS].to_numpy(float) if list(frame.columns) == ['StudyInstanceUID', *TARGETS] else None
    ok = v is not None and len(frame) >= 2 and frame.StudyInstanceUID.is_unique and bool(np.isfinite(v).all() and ((v >= 0) & (v <= 1)).all())
    return ok and (uids is None or frame.StudyInstanceUID.tolist() == uids)
PUBLIC_UIDS = ['1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191', '1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776', '1.2.826.0.1.3680043.8.498.10067514707072572280263481548497591402']  # the 3 public sample studies
SIXWAY_SRC_SHA = 'e2c0dad12aaf00b40d412a2c0619bc02a573dac046f699179e5c84c0b21f9b9a'
SIXWAY_REF_SHA = '9e42cd0844c2dc7dd7bcc22f995e849c368b060abc3565665d252b876fc55e49'  # 359192097 on the 3 public studies
SIXWAY_B64 = 'ZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lLHRpbWV6b25lCmltcG9ydCBjc3YsaGFzaGxpYixpbXBvcnRsaWIudXRpbCxqc29uLG9zLHNodXRpbCxzaWduYWwsc3VicHJvY2VzcyxzeXMsdGFyZmlsZSx0aHJlYWRpbmcsdGltZQpTVEFSVD10aW1lLnBlcmZfY291bnRlcigpO0VOVFJZPWRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpLmlzb2Zvcm1hdCgpCmRlZiBwaGFzZShuYW1lLCoqa3cpOnByaW50KCdSRVNJREVOVF9QSEFTRScsanNvbi5kdW1wcyhkaWN0KHBoYXNlPW5hbWUsc2Vjb25kcz10aW1lLnBlcmZfY291bnRlcigpLVNUQVJULHV0Yz1kYXRldGltZS5ub3codGltZXpvbmUudXRjKS5pc29mb3JtYXQoKSwqKmt3KSksZmx1c2g9VHJ1ZSkKZGVmIHNoYShwKToKIHdpdGggUGF0aChwKS5vcGVuKCdyYicpIGFzIGY6cmV0dXJuIGhhc2hsaWIuZmlsZV9kaWdlc3QoZiwnc2hhMjU2JykuaGV4ZGlnZXN0KCkKZGVmIHJlYWQocCk6cmV0dXJuIGpzb24ubG9hZHMoUGF0aChwKS5yZWFkX3RleHQoKSkKZGVmIHNhdmUocCx2KTpQYXRoKHApLndyaXRlX3RleHQoanNvbi5kdW1wcyh2LGluZGVudD0yKSsnXG4nKQpwaGFzZSgnRU5UUlknKQpST09UPVBhdGgoJy9rYWdnbGUvaW5wdXQnKTtXT1JLPVBhdGgoJy9rYWdnbGUvd29ya2luZycpO1RNUD1QYXRoKCcvdG1wL3JzbmEtc2l4d2F5LXJlc2lkZW50LTIwMjYxMDE3Jyk7VE1QLm1rZGlyKCkKU0xVRz0ncnNuYS1zaXh3YXktcmVzaWRlbnQtcnVudGltZS0yMDI2MTAxNyc7RVhQRUNURUQ9JzA4OWM1MDI3MWRjYjc2NWVlMzM5ZWIwY2ZmOTI0NmMzNDQwOTMwNGYxYTA5ZGJlNzNjNThlOGMwNWZkZDhiMjgnCm1hdGNoZXM9W3AgZm9yIHAgaW4gW1JPT1QvU0xVRyxST09ULydkYXRhc2V0cy9pbHlha29sYicvU0xVR10gaWYgKHAvJ1JFTEVBU0VfUEFDS0VULmpzb24nKS5pc19maWxlKCkgYW5kIHNoYShwLydSRUxFQVNFX1BBQ0tFVC5qc29uJyk9PUVYUEVDVEVEXQphc3NlcnQgbGVuKG1hdGNoZXMpPT0xO0RTPW1hdGNoZXNbMF07cmVsZWFzZT1yZWFkKERTLydSRUxFQVNFX1BBQ0tFVC5qc29uJykKZm9yIG5hbWUscGluIGluIHJlbGVhc2VbJ2ZpbGVzJ10uaXRlbXMoKTphc3NlcnQgc2hhKERTL25hbWUpPT1waW4sbmFtZQptb2RlbHM9cmVsZWFzZVsnbW9kZWxzJ107YXNzZXJ0IGxlbihtb2RlbHMpPT0xMApmb3IgaXRlbSBpbiBtb2RlbHM6CiBjYW5kaWRhdGVzPVtwIGZvciBwIGluIFtST09UL2l0ZW1bJ2RhdGFzZXRfc2x1ZyddLFJPT1QvJ2RhdGFzZXRzL2lseWFrb2xiJy9pdGVtWydkYXRhc2V0X3NsdWcnXV0gaWYgcC5pc19kaXIoKV07YXNzZXJ0IGxlbihjYW5kaWRhdGVzKT09MQogaXRlbVsnbW91bnQnXT1zdHIoY2FuZGlkYXRlc1swXSk7cj1yZWFkKGNhbmRpZGF0ZXNbMF0vaXRlbVsncmVsZWFzZV9maWxlJ10pO2Fzc2VydCBzaGEoY2FuZGlkYXRlc1swXS9pdGVtWydyZWxlYXNlX2ZpbGUnXSk9PWl0ZW1bJ3JlbGVhc2Vfc2hhMjU2J10KIGl0ZW1bJ3RyYWluaW5nX2lkZW50aXR5J109ci5nZXQoJ3RyYWluaW5nX2lkZW50aXR5JykgaWYgaXRlbVsna2V5J109PSdtaXJyb3InIGVsc2UgclsnbW9kZWxzJ11bMF1bJ3RyYWluaW5nX2lkZW50aXR5J10KIGl0ZW1bJ2NoZWNrcG9pbnQnXT1zdHIoY2FuZGlkYXRlc1swXS8oaXRlbVsna2V5J10rJy5wdCcgaWYgaXRlbVsnYmluZGluZyddIGVsc2UgJ2NoZWNrcG9pbnQucHQnKSkKIGl0ZW1bJ29yaWdpbmFsX3JlbGVhc2UnXT1yCkE9bmV4dChtIGZvciBtIGluIG1vZGVscyBpZiBtWydrZXknXT09J2FkYXB0aXZlJyk7TT1uZXh0KG0gZm9yIG0gaW4gbW9kZWxzIGlmIG1bJ2tleSddPT0nbWlycm9yJykKc3lzLnBhdGguaW5zZXJ0KDAsQVsnbW91bnQnXSk7ZnJvbSBrYWdnbGVfbW91bnRzIGltcG9ydCBjb21wZXRpdGlvbl9wYXJlbnQKQ09NUD1jb21wZXRpdGlvbl9wYXJlbnQoUk9PVCxyZWxlYXNlWydjb21wZXRpdGlvbiddKTtzYW1wbGVfcm93cz1saXN0KGNzdi5EaWN0UmVhZGVyKChDT01QLydzYW1wbGVfc3VibWlzc2lvbi5jc3YnKS5vcGVuKCkpKTt1aWRzPVtyWydTdHVkeUluc3RhbmNlVUlEJ10gZm9yIHIgaW4gc2FtcGxlX3Jvd3NdCnJlZl91aWRzPVtyWydTdHVkeUluc3RhbmNlVUlEJ10gZm9yIHIgaW4gY3N2LkRpY3RSZWFkZXIoKERTLydyZWZfYWRhcHRpdmUuY3N2Jykub3BlbigpKV07U0FNUExFX1JVTj11aWRzPT1yZWZfdWlkcwphc3NlcnQgbGVuKHVpZHMpPj0yIGFuZCBsZW4odWlkcyk9PWxlbihzZXQodWlkcykpClRPVEFMX0xJTUlUPTE4MDAgaWYgU0FNUExFX1JVTiBlbHNlIDEzNDAwCnNpZ25hbC5zaWduYWwoc2lnbmFsLlNJR0FMUk0sbGFtYmRhICphOihfIGZvciBfIGluICgpKS50aHJvdyhUaW1lb3V0RXJyb3IoJ1Jlc2lkZW50IHRvdGFscnVudGltZWNhcCcpKSk7c2lnbmFsLmFsYXJtKFRPVEFMX0xJTUlUKQpkZWYgdW5wYWNrKGl0ZW0sbmFtZSxkZXN0KToKIG1vdW50PVBhdGgoaXRlbVsnbW91bnQnXSk7YXJjaGl2ZT1tb3VudC9uYW1lO2Rlc3QubWtkaXIoKQogaWYgYXJjaGl2ZS5pc19maWxlKCk6CiAgYXNzZXJ0IHNoYShhcmNoaXZlKT09aXRlbVsnb3JpZ2luYWxfcmVsZWFzZSddWydmaWxlcyddW25hbWVdCiAgd2l0aCB0YXJmaWxlLm9wZW4oYXJjaGl2ZSkgYXMgdGFyOnRhci5leHRyYWN0YWxsKGRlc3QsZmlsdGVyPSdkYXRhJykKIGVsc2U6CiAgZGVzdC5ybWRpcigpO3NodXRpbC5jb3B5dHJlZShtb3VudC9QYXRoKG5hbWUpLnN0ZW0sZGVzdCkKIHBpbnM9aXRlbVsnb3JpZ2luYWxfcmVsZWFzZSddWydhcmNoaXZlX2ZpbGVzJ11bbmFtZV0KIGFzc2VydCB7c3RyKHAucmVsYXRpdmVfdG8oZGVzdCkpIGZvciBwIGluIGRlc3Qucmdsb2IoJyonKSBpZiBwLmlzX2ZpbGUoKX09PXNldChwaW5zKQogZm9yIG5hbWUscGluIGluIHBpbnMuaXRlbXMoKTphc3NlcnQgc2hhKGRlc3QvbmFtZSk9PXBpbixuYW1lCiByZXR1cm4gZGVzdApTT1VSQ0U9dW5wYWNrKEEsJ3NvdXJjZS50YXInLFRNUC8nY2Fub25pY2FsX3NvdXJjZScpO1JVTlRJTUU9dW5wYWNrKEEsJ3IzX3J1bnRpbWUudGFyJyxUTVAvJ2Nhbm9uaWNhbF9ydW50aW1lJyk7V0hFRUxTPXVucGFjayhBLCd3aGVlbHMudGFyJyxUTVAvJ3doZWVscycpClJBV19TT1VSQ0U9dW5wYWNrKE0sJ3NvdXJjZS50YXInLFRNUC8ncmF3X3NvdXJjZScpO1JBV19SVU5USU1FPXVucGFjayhNLCdydW50aW1lLnRhcicsVE1QLydyYXdfcnVudGltZScpCmZvciBtb2RlbCBpbiBtb2RlbHM6CiBpZiBtb2RlbFsna2V5J109PSdtaXJyb3InOmNvbnRpbnVlCiBmb3IgbmFtZSxwaW4gaW4gbW9kZWxbJ29yaWdpbmFsX3JlbGVhc2UnXVsnYXJjaGl2ZV9maWxlcyddWydzb3VyY2UudGFyJ10uaXRlbXMoKTphc3NlcnQgc2hhKFNPVVJDRS9uYW1lKT09cGluLChtb2RlbFsna2V5J10sbmFtZSkKYXNzZXJ0IG5vdCAoUkFXX1NPVVJDRS8nbGF0ZXJhbGl0eScpLmV4aXN0cygpIGFuZCBub3QgbGlzdChSQVdfU09VUkNFLnJnbG9iKCcqLnB0JykpCkRFUFM9VE1QLydkZXBzJztERVBTLm1rZGlyKCkKZnJvbSBwYWNrYWdpbmcudGFncyBpbXBvcnQgc3lzX3RhZ3MKZnJvbSBwYWNrYWdpbmcudXRpbHMgaW1wb3J0IHBhcnNlX3doZWVsX2ZpbGVuYW1lCnRhZ3M9c2V0KHN5c190YWdzKCkpO3doZWVscz1bcCBmb3IgcCBpbiBzb3J0ZWQoV0hFRUxTLmdsb2IoJyoud2hsJykpIGlmIHBhcnNlX3doZWVsX2ZpbGVuYW1lKHAubmFtZSlbM10mdGFnc10KYXNzZXJ0IGFueShwLm5hbWUuc3RhcnRzd2l0aCgndGltbS0nKSBmb3IgcCBpbiB3aGVlbHMpIGFuZCBhbnkocC5uYW1lLnN0YXJ0c3dpdGgoJ3B5ZGljb20tJykgZm9yIHAgaW4gd2hlZWxzKQpzdWJwcm9jZXNzLnJ1bihbc3lzLmV4ZWN1dGFibGUsJy1tJywncGlwJywnaW5zdGFsbCcsJy0tZGlzYWJsZS1waXAtdmVyc2lvbi1jaGVjaycsJy0tbm8taW5kZXgnLCctLW5vLWRlcHMnLCctLXRhcmdldCcsc3RyKERFUFMpLCptYXAoc3RyLHdoZWVscyldLGNoZWNrPVRydWUpCmZvciBmb2xkZXIsbmFtZSxuZXcgaW4gWyhSVU5USU1FLCdwcm9jZXNzX2NvbnRyb2xsZXIucHknLCdyZXNpZGVudF9jb250cm9sbGVyLnB5JyksKFJBV19SVU5USU1FLCdpc29sYXRlZF93b3JrZXIucHknLCdtaXJyb3JfcGFja2VkX3dvcmtlci5weScpXToKIHNodXRpbC5jb3B5ZmlsZShEUy9uZXcsZm9sZGVyL25hbWUpO21hbmlmZXN0PXJlYWQoZm9sZGVyLydTT1VSQ0VfTUFOSUZFU1QuanNvbicpO21hbmlmZXN0WydydW50aW1lJ11bbmFtZV09c2hhKGZvbGRlci9uYW1lKTtzYXZlKGZvbGRlci8nU09VUkNFX01BTklGRVNULmpzb24nLG1hbmlmZXN0KQpwaGFzZSgnQk9PVFNUUkFQX0NPTVBMRVRFJyxkZXBlbmRlbmN5X2luc3RhbGxzPTEsY2Fub25pY2FsX3NvdXJjZV9leHRyYWN0aW9ucz0xLHJhd19zb3VyY2VfZXh0cmFjdGlvbnM9MSkKaW5wPVRNUC8nb2ZmaWNpYWxfaW5wdXQnO2lucC5ta2RpcigpCmZvciBuYW1lIGluIFsnc2FtcGxlX3N1Ym1pc3Npb24uY3N2JywndGVzdF9zZXJpZXMuY3N2J106c2h1dGlsLmNvcHlmaWxlKENPTVAvbmFtZSxpbnAvbmFtZSkKKGlucC8ndGVzdF9zZXJpZXMnKS5zeW1saW5rX3RvKENPTVAvJ3Rlc3Rfc2VyaWVzJyx0YXJnZXRfaXNfZGlyZWN0b3J5PVRydWUpCnNlcmllc19yb3dzPWxpc3QoY3N2LkRpY3RSZWFkZXIoKENPTVAvJ3Rlc3Rfc2VyaWVzLmNzdicpLm9wZW4oKSkpO3NpemVzPXt1OjAgZm9yIHUgaW4gdWlkc30KZm9yIHIgaW4gc2VyaWVzX3Jvd3M6c2l6ZXNbclsnU3R1ZHlJbnN0YW5jZVVJRCddXSs9MQphbGxvY2F0aW9uPVtbXSxbXV07Y29zdD1bMCwwXQpmb3IgdWlkIGluIHNvcnRlZCh1aWRzLGtleT1sYW1iZGEgdTooLXNpemVzW3VdLHUpKToKIG93bmVyPW1pbihyYW5nZSgyKSxrZXk9bGFtYmRhIGk6KGNvc3RbaV0sbGVuKGFsbG9jYXRpb25baV0pLGkpKTthbGxvY2F0aW9uW293bmVyXS5hcHBlbmQodWlkKTtjb3N0W293bmVyXSs9c2l6ZXNbdWlkXQphc3NlcnQgYWxsKGFsbG9jYXRpb24pClJBV19CQU5LPVRNUC8ncmF3X2JhbmsnO1JBV19CQU5LLm1rZGlyKCkKZW52PWRpY3Qob3MuZW52aXJvbixQWVRIT05QQVRIPW9zLnBhdGhzZXAuam9pbihbc3RyKERTKSxzdHIoREVQUyldKSxQWVRIT05OT1VTRVJTSVRFPScxJyxQWVRIT05ET05UV1JJVEVCWVRFQ09ERT0nMScsT01QX05VTV9USFJFQURTPScxJyxNS0xfTlVNX1RIUkVBRFM9JzEnLE9QRU5CTEFTX05VTV9USFJFQURTPScxJyxOVU1FWFBSX05VTV9USFJFQURTPScxJyxSU05BX1NIQVJFRF9SQVdfQkFOSz1zdHIoUkFXX0JBTkspKQplbnYucG9wKCdSU05BX1NIQVJFRF9JTlBVVF9CQU5LJyxOb25lKQpndWFyZD1SQVdfUlVOVElNRS8ncHJvY2Vzc19ndWFyZC5weSc7YXNzZXJ0IHNoYShndWFyZCk9PScwMmMxMDJlNGI0ZjI1ZmJjMTZlODU0YzZhOWM4ZTRlZmRlODk1MTQ1YmU0Y2UzNzU5MWI2NTkxNzYxMjA3MWI3JwpkZWYgcnVuKGFyZ3MsY2hpbGRfZW52LGxvZyk6CiBzcGVjPWltcG9ydGxpYi51dGlsLnNwZWNfZnJvbV9maWxlX2xvY2F0aW9uKCdyc25hX3Jlc2lkZW50X2d1YXJkJyxndWFyZCk7Zz1pbXBvcnRsaWIudXRpbC5tb2R1bGVfZnJvbV9zcGVjKHNwZWMpO3NwZWMubG9hZGVyLmV4ZWNfbW9kdWxlKGcpCiB3aXRoIFBhdGgobG9nKS5vcGVuKCd3JyxidWZmZXJpbmc9MSkgYXMgZjoKICBwcm9jZXNzPXN1YnByb2Nlc3MuUG9wZW4oYXJncyxlbnY9Y2hpbGRfZW52LHN0ZG91dD1mLHN0ZGVycj1zdWJwcm9jZXNzLlNURE9VVCxzdGFydF9uZXdfc2Vzc2lvbj1UcnVlKQogIHRyeToKICAgZGVhZGxpbmU9dGltZS5tb25vdG9uaWMoKStUT1RBTF9MSU1JVC0odGltZS5wZXJmX2NvdW50ZXIoKS1TVEFSVCkKICAgd2hpbGUgcHJvY2Vzcy5wb2xsKCkgaXMgTm9uZToKICAgIHRyeTpwcm9jZXNzLndhaXQodGltZW91dD0xNSkKICAgIGV4Y2VwdCBzdWJwcm9jZXNzLlRpbWVvdXRFeHBpcmVkOgogICAgIHBoYXNlKCdDT05UUk9MTEVSX1JVTk5JTkcnLGNvbnRyb2xsZXI9UGF0aChsb2cpLnN0ZW0pCiAgICAgYXNzZXJ0IHRpbWUubW9ub3RvbmljKCk8ZGVhZGxpbmUKICAgYXNzZXJ0IHByb2Nlc3MucmV0dXJuY29kZT09MCwoc3RyKGxvZykscHJvY2Vzcy5yZXR1cm5jb2RlKQogIGZpbmFsbHk6CiAgIGcuY2xlYW51cChwcm9jZXNzLnBpZCkKICAgdHJ5OnByb2Nlc3Mud2FpdCh0aW1lb3V0PTMpCiAgIGV4Y2VwdCBzdWJwcm9jZXNzLlRpbWVvdXRFeHBpcmVkOnByb2Nlc3Mua2lsbCgpO3Byb2Nlc3Mud2FpdCh0aW1lb3V0PTMpCnJlc291cmNlPXJlYWQoRFMvJ0FEQVBUSVZFX1dPUlNUQ0FTRV9SRVNPVVJDRV9SRUZFUkVOQ0UuanNvbicpCmNmZz1yZWFkKFJVTlRJTUUvJ1JFQ0lQRS5qc29uJyk7Y2ZnLnVwZGF0ZShzb3VyY2U9c3RyKFNPVVJDRSkscnVudGltZT1zdHIoUlVOVElNRSksZGF0YXNldD1BWydtb3VudCddLGNoZWNrcG9pbnQ9QVsnY2hlY2twb2ludCddLHNvdXJjZV9tYW5pZmVzdD1zdHIoUlVOVElNRS8nU09VUkNFX01BTklGRVNULmpzb24nKSxpbnB1dD1zdHIoaW5wKSxvdXRwdXQ9c3RyKFdPUksvJ2Nhbm9uaWNhbCcpLGFsbG9jYXRpb249YWxsb2NhdGlvbix0cmFpbmluZ19pZGVudGl0eT1BWyd0cmFpbmluZ19pZGVudGl0eSddLG1vZGVscz1be2s6diBmb3Igayx2IGluIG0uaXRlbXMoKSBpZiBrIT0nb3JpZ2luYWxfcmVsZWFzZSd9IGZvciBtIGluIG1vZGVsc10scGFja2VkX3Jvb3Q9c3RyKFRNUC8ncGFja2VkX3dlaWdodHMnKSxwYWNrZWRfbW9kZWxzX21hbmlmZXN0PXN0cihUTVAvJ1BBQ0tFRF9NT0RFTFMuanNvbicpLGFkYXB0aXZlX3Jlc291cmNlX3JlZmVyZW5jZV9wZWFrX3Jlc2VydmVkX2J5dGVzPW1heChyWydwZWFrX3Jlc2VydmVkX2J5dGVzJ10gZm9yIHIgaW4gcmVzb3VyY2VbJ3JlY29yZHMnXSksYWRhcHRpdmVfcmVzb3VyY2VfcmVmZXJlbmNlX3NoYTI1Nj1zaGEoRFMvJ0FEQVBUSVZFX1dPUlNUQ0FTRV9SRVNPVVJDRV9SRUZFUkVOQ0UuanNvbicpKQpjb25maWc9VE1QLydjYW5vbmljYWxfY29uZmlnLmpzb24nO3NhdmUoY29uZmlnLGNmZykKcGhhc2UoJ0NBTk9OSUNBTF9TVEFSVCcscmVzaWRlbnRfY2hlY2twb2ludHNfcGVyX2dwdT05KQpydW4oW3N5cy5leGVjdXRhYmxlLCctQicsJy11JyxzdHIoUlVOVElNRS8ncHJvY2Vzc19jb250cm9sbGVyLnB5JyksJy0tY29uZmlnJyxzdHIoY29uZmlnKV0sZGljdChlbnYsUlNOQV9DT01QT05FTlRfS0VZPSdhZGFwdGl2ZScpLFdPUksvJ0NBTk9OSUNBTF9DT05UUk9MTEVSLmxvZycpCnBoYXNlKCdDQU5PTklDQUxfQ09NUExFVEUnKQpwYWNrZWQ9cmVhZChUTVAvJ1BBQ0tFRF9NT0RFTFMuanNvbicpO2Fzc2VydCBsZW4ocGFja2VkKT09MTAKcGFja19xYT17bVsna2V5J106cmVhZChQYXRoKG1bJ3BhY2tlZF93ZWlnaHRzJ10pLydIRUFERVIuanNvbicpIGZvciBtIGluIHBhY2tlZH0Kc2F2ZShXT1JLLydNT0RFTF9QQUNLX1FBLmpzb24nLHtrOnthOnYgZm9yIGEsdiBpbiByLml0ZW1zKCkgaWYgYSE9J3RlbnNvcnMnIGFuZCBhIT0naWRlbnRpdHknfSBmb3IgayxyIGluIHBhY2tfcWEuaXRlbXMoKX0pCm1pcnJvcj1uZXh0KG0gZm9yIG0gaW4gcGFja2VkIGlmIG1bJ2tleSddPT0nbWlycm9yJyk7cmF3X2NmZz1yZWFkKFJBV19SVU5USU1FLydSRUNJUEUuanNvbicpCnJhd19jZmcudXBkYXRlKHNvdXJjZT1zdHIoUkFXX1NPVVJDRSksZGF0YXNldD1NWydtb3VudCddLGNoZWNrcG9pbnQ9TVsnY2hlY2twb2ludCddLHNvdXJjZV9tYW5pZmVzdD1zdHIoUkFXX1JVTlRJTUUvJ1NPVVJDRV9NQU5JRkVTVC5qc29uJyksaW5wdXQ9c3RyKGlucCksb3V0cHV0PXN0cihXT1JLLydtaXJyb3JfcnVuJyksYWxsb2NhdGlvbj1hbGxvY2F0aW9uLHRyYWluaW5nX2lkZW50aXR5PU1bJ3RyYWluaW5nX2lkZW50aXR5J10scGFja2VkX3dlaWdodHM9bWlycm9yWydwYWNrZWRfd2VpZ2h0cyddLHBhY2tlZF9jaGVja3BvaW50X3NoYTI1Nj1NWydjaGVja3BvaW50X3NoYTI1NiddKQpyYXdfY29uZmlnPVRNUC8ncmF3X2NvbmZpZy5qc29uJztzYXZlKHJhd19jb25maWcscmF3X2NmZykKcGhhc2UoJ01JUlJPUl9SQVdfU1RBUlQnKQpydW4oW3N5cy5leGVjdXRhYmxlLCctQicsJy11JyxzdHIoUkFXX1JVTlRJTUUvJ3Byb2Nlc3NfY29udHJvbGxlci5weScpLCctLWNvbmZpZycsc3RyKHJhd19jb25maWcpXSxkaWN0KGVudixSU05BX0NPTVBPTkVOVF9LRVk9J21pcnJvcicpLFdPUksvJ01JUlJPUl9DT05UUk9MTEVSLmxvZycpCnBoYXNlKCdNSVJST1JfUkFXX0NPTVBMRVRFJykKc3lzLnBhdGhbOjBdPVtzdHIoREVQUyksc3RyKERTKV0KaW1wb3J0IG51bXB5IGFzIG5wLHBhbmRhcyBhcyBwZApmcm9tIGpvaW50MzZfcmFua2VkIGltcG9ydCByYW5rX2ZyYW1lLFRBUkdFVFMKc2FtcGxlPXBkLnJlYWRfY3N2KENPTVAvJ3NhbXBsZV9zdWJtaXNzaW9uLmNzdicsZHR5cGU9eydTdHVkeUluc3RhbmNlVUlEJzpzdHJ9KTttb2RlbF9mcmFtZXM9e307cGFyaXR5PXt9O21vZGVsX2Rpcj1XT1JLLydtb2RlbHMnO21vZGVsX2Rpci5ta2RpcigpCmZvciBpdGVtIGluIG1vZGVsczoKIGtleT1pdGVtWydrZXknXTtwYXRoPVdPUksvJ21pcnJvcl9ydW4vY29yZV9wcm9iYWJpbGl0aWVzLmNzdicgaWYga2V5PT0nbWlycm9yJyBlbHNlIFdPUksvJ2Nhbm9uaWNhbCcvKGtleSsnLmNzdicpCiBmcmFtZT1wZC5yZWFkX2NzdihwYXRoLGR0eXBlPXsnU3R1ZHlJbnN0YW5jZVVJRCc6c3RyfSkKIGlmIGtleT09J2pvaW50JzpmcmFtZT1yYW5rX2ZyYW1lKGZyYW1lKQogYXNzZXJ0IGZyYW1lLlN0dWR5SW5zdGFuY2VVSUQudG9saXN0KCk9PXVpZHMgYW5kIGxpc3QoZnJhbWUuY29sdW1ucyk9PWxpc3Qoc2FtcGxlLmNvbHVtbnMpCiBmcmFtZS50b19jc3YobW9kZWxfZGlyLyhrZXkrJy5jc3YnKSxpbmRleD1GYWxzZSkKIHBhcnNlZD1wZC5yZWFkX2Nzdihtb2RlbF9kaXIvKGtleSsnLmNzdicpLGR0eXBlPXsnU3R1ZHlJbnN0YW5jZVVJRCc6c3RyfSxmbG9hdF9wcmVjaXNpb249J3JvdW5kX3RyaXAnKTttb2RlbF9mcmFtZXNba2V5XT1wYXJzZWQKIGlmIFNBTVBMRV9SVU46CiAgcmVmZXJlbmNlPXBkLnJlYWRfY3N2KERTL2l0ZW1bJ3JlZmVyZW5jZV9maWxlJ10sZHR5cGU9eydTdHVkeUluc3RhbmNlVUlEJzpzdHJ9LGZsb2F0X3ByZWNpc2lvbj0ncm91bmRfdHJpcCcpCiAgZGVsdGE9bnAuYWJzKHBhcnNlZFtUQVJHRVRTXS50b19udW1weShmbG9hdCktcmVmZXJlbmNlW1RBUkdFVFNdLnRvX251bXB5KGZsb2F0KSk7YXNzZXJ0IHBhcnNlZC5lcXVhbHMocmVmZXJlbmNlKSwoa2V5LGZsb2F0KGRlbHRhLm1heCgpKSkKICBwYXJpdHlba2V5XT17J3N0YXR1cyc6J1BBU1MnLCdtYXhfYWJzb2x1dGVfZGlmZmVyZW5jZSc6ZmxvYXQoZGVsdGEubWF4KCkpLCdjc3ZfYnl0ZXNfZXF1YWwnOnNoYShtb2RlbF9kaXIvKGtleSsnLmNzdicpKT09aXRlbVsncmVmZXJlbmNlX3NoYTI1NiddfQpmb2xkPW5wLm1lYW4obnAuc3RhY2soW21vZGVsX2ZyYW1lc1snZm9sZCcrc3RyKGkpXVtUQVJHRVRTXS50b19udW1weShmbG9hdCkgZm9yIGkgaW4gcmFuZ2UoNSldKSxheGlzPTApCmNvbXBvbmVudHM9eydmaXZlZm9sZCc6Zm9sZCwqKntrOm1vZGVsX2ZyYW1lc1trXVtUQVJHRVRTXS50b19udW1weShmbG9hdCkgZm9yIGsgaW4gWydhZGFwdGl2ZScsJ3JlcGFpcicsJ2NvcmUnLCdtaXJyb3InLCdqb2ludCddfX0KaWYgU0FNUExFX1JVTjphc3NlcnQgbnAuYXJyYXlfZXF1YWwoZm9sZCxwZC5yZWFkX2NzdihEUy8ncmVmX2ZpdmVmb2xkLmNzdicsZmxvYXRfcHJlY2lzaW9uPSdyb3VuZF90cmlwJylbVEFSR0VUU10udG9fbnVtcHkoZmxvYXQpKQpmaW5hbF92YWx1ZXM9bnAubWVhbihucC5zdGFjayhsaXN0KGNvbXBvbmVudHMudmFsdWVzKCkpKSxheGlzPTApO2Fzc2VydCBucC5pc2Zpbml0ZShmaW5hbF92YWx1ZXMpLmFsbCgpIGFuZCAoKGZpbmFsX3ZhbHVlcz49MCkmKGZpbmFsX3ZhbHVlczw9MSkpLmFsbCgpCmZpbmFsPXBkLkRhdGFGcmFtZShmaW5hbF92YWx1ZXMsY29sdW1ucz1UQVJHRVRTKTtmaW5hbC5pbnNlcnQoMCwnU3R1ZHlJbnN0YW5jZVVJRCcsdWlkcyk7ZmluYWwudG9fY3N2KFdPUksvJ3N1Ym1pc3Npb24uY3N2JyxpbmRleD1GYWxzZSkKY29tcG9uZW50X2Rpcj1XT1JLLydjb21wb25lbnRzJztjb21wb25lbnRfZGlyLm1rZGlyKCkKZm9yIGtleSx2YWx1ZXMgaW4gY29tcG9uZW50cy5pdGVtcygpOgogZj1wZC5EYXRhRnJhbWUodmFsdWVzLGNvbHVtbnM9VEFSR0VUUyk7Zi5pbnNlcnQoMCwnU3R1ZHlJbnN0YW5jZVVJRCcsdWlkcyk7Zi50b19jc3YoY29tcG9uZW50X2Rpci8oa2V5KycuY3N2JyksaW5kZXg9RmFsc2UpCmF1ZGl0PXJlYWQoV09SSy8nY2Fub25pY2FsL1BST0ZJTEVfQVVESVQuanNvbicpO3Jhd19hdWRpdD1yZWFkKFdPUksvJ21pcnJvcl9ydW4vUFJPRklMRV9BVURJVC5qc29uJyk7YXNzZXJ0IGF1ZGl0WydzdGF0dXMnXT09cmF3X2F1ZGl0WydzdGF0dXMnXT09J1BBU1MnCmZvciB3b3JrZXIgaW4gcmF3X2F1ZGl0Wyd3b3JrZXJzJ106YXNzZXJ0IHdvcmtlclsnb3JpZW50YXRpb25fbW9kZSddPT13b3JrZXJbJ29yaWVudGF0aW9uX3N0YXRlJ109PSdyYXdfYWNxdWlyZWQnIGFuZCB3b3JrZXJbJ29yaWVudGF0aW9uX3BvbGljeSddPT0nZXhwbGljaXQtcmF3LWFjcXVpcmVkLW5vLWNhbm9uaWNhbGl6YXRpb24tdjEnCmFzc2VydCBhbGwod1snc3ludGhldGljX2ZvcndhcmRzJ109PTAgYW5kIGxlbih3Wydtb2RlbF9sb2FkcyddKT09OSBhbmQgYWxsKG1bJ3N0cmljdF9sb2FkcyddPT1tWydkZXZpY2VfY29waWVzJ109PTEgZm9yIG0gaW4gd1snbW9kZWxfbG9hZHMnXS52YWx1ZXMoKSkgZm9yIHcgaW4gYXVkaXRbJ3dvcmtlcnMnXSkKZXZlbnRzPVtqc29uLmxvYWRzKHMpIGZvciBzIGluIChSQVdfQkFOSy8nRVZFTlRTLmpzb25sJykucmVhZF90ZXh0KCkuc3BsaXRsaW5lcygpXTtidWlsZHM9W2UgZm9yIGUgaW4gZXZlbnRzIGlmIGVbJ2tpbmQnXT09J3NvdXJjZV9idWlsZCddO2Jhc2U9W2UgZm9yIGUgaW4gZXZlbnRzIGlmIGVbJ2tpbmQnXT09J2Nhbm9uaWNhbF9pbnB1dF9idWlsZCddO2ZvcndhcmRzPVtlIGZvciBlIGluIGV2ZW50cyBpZiBlWydraW5kJ109PSdyZXNpZGVudF9mb3J3YXJkJ10KYXNzZXJ0IGxlbihidWlsZHMpPT1sZW4oe2VbJ3BhdGgnXSBmb3IgZSBpbiBidWlsZHN9KSBhbmQgYWxsKGVbJ3NvdXJjZV9yZWFkcyddPT1lWydwaXhlbF9kZWNvZGVzJ109PTEgYW5kIG5vdCBlWydkZWNvZGVfZXJyb3InXSBmb3IgZSBpbiBidWlsZHMpCmFzc2VydCBsZW4oYmFzZSk9PWxlbih7KGVbJ3VpZCddLGVbJ3JvdXRlJ10pIGZvciBlIGluIGJhc2V9KSBhbmQgbGVuKGZvcndhcmRzKT09OSpsZW4odWlkcykKYXNzZXJ0IG5vdCBhbnkoZS5nZXQoJ2NvbXBvbmVudCcpPT0nbWlycm9yJyBhbmQgZVsna2luZCddPT0nY2Fub25pY2FsX2lucHV0X2J1aWxkJyBmb3IgZSBpbiBldmVudHMpCmV4dHJhcz17clsndWlkJ106c3VtKGFbJ2FkYXB0aXZlX2V4dGVuc2lvbiddWyd3aW5kb3dzJ10gZm9yIGEgaW4gclsnYWRhcHRpdmVfZXh0cmEnXSkgZm9yIHcgaW4gYXVkaXRbJ3dvcmtlcnMnXSBmb3IgciBpbiB3WydyZWNvcmRzJ119CmlmIFNBTVBMRV9SVU46YXNzZXJ0IGxlbihidWlsZHMpPT01NTcgYW5kIFtleHRyYXNbdV0gZm9yIHUgaW4gdWlkc109PVsyNCwxMiwwXQpyZWFkX3FhPXsnc3RhdHVzJzonUEFTUycsJ3VuaXF1ZV9vcmlnaW5hbF9kaWNvbXMnOmxlbihidWlsZHMpLCdvcmlnaW5hbF9zb3VyY2VfcmVhZHMnOnN1bShlLmdldCgnc291cmNlX3JlYWRzJywwKSBmb3IgZSBpbiBldmVudHMpLCdyYXdfcGl4ZWxfYXJyYXlzJzpzdW0oZS5nZXQoJ3BpeGVsX2RlY29kZXMnLDApIGZvciBlIGluIGV2ZW50cyksJ2Nhbm9uaWNhbF9yb3V0ZXNfcHJlcGFyZWQnOmxlbihiYXNlKSwnY2Fub25pY2FsX3RlbnNvcl9yZXVzZXMnOnN1bShlWydyb3V0ZXMnXSBmb3IgZSBpbiBmb3J3YXJkcyBpZiBlWydzaGFyZWRfY2Fub25pY2FsX2lucHV0J10pLCdkZXBlbmRlbmN5X2luc3RhbGxzJzoxLCdjaGVja3BvaW50X3NvdXJjZV9yZWFkcyc6c3VtKHJbJ3NvdXJjZV9yZWFkcyddIGZvciByIGluIHBhY2tfcWEudmFsdWVzKCkpLCdjaGVja3BvaW50X2Rlc2VyaWFsaXphdGlvbnMnOnN1bShyWydjaGVja3BvaW50X2Rlc2VyaWFsaXphdGlvbnMnXSBmb3IgciBpbiBwYWNrX3FhLnZhbHVlcygpKSwnY2Fub25pY2FsX0dQVV9wcm9jZXNzZXMnOjIsJ3Jhd19HUFVfcHJvY2Vzc2VzJzoyLCdjYW5vbmljYWxfbW9kZWxfc3RyaWN0X2xvYWRzX3Blcl9HUFUnOjksJ3N5bnRoZXRpY19mb3J3YXJkcyc6MCwnYWRhcHRpdmVfZXh0cmFfd2luZG93cyc6ZXh0cmFzLCdyYXdfYmFua19ieXRlcyc6c3VtKHAuc3RhdCgpLnN0X3NpemUgZm9yIHAgaW4gUkFXX0JBTksuZ2xvYignKi56cGsnKSl9CnNhdmUoV09SSy8nUkVBRF9SRVVTRV9RQS5qc29uJyxyZWFkX3FhKTtzaHV0aWwuY29weWZpbGUoUkFXX0JBTksvJ0VWRU5UUy5qc29ubCcsV09SSy8nU0hBUkVEX0JBTktfRVZFTlRTLmpzb25sJykKc2F2ZShXT1JLLydDT01QT05FTlRfUEFSSVRZX1FBLmpzb24nLHsnc3RhdHVzJzonUEFTUycgaWYgU0FNUExFX1JVTiBlbHNlICdOT1RfUFVCTElDX1NBTVBMRScsJ21vZGVscyc6cGFyaXR5LCdmaXZlZm9sZCc6J1BBU1MnIGlmIFNBTVBMRV9SVU4gZWxzZSAnTk9UX1BVQkxJQ19TQU1QTEUnLCdvdXRlcl93ZWlnaHRzJzp7azonMS82JyBmb3IgayBpbiBjb21wb25lbnRzfSwnZWZmZWN0aXZlX2ZvbGRfd2VpZ2h0cyc6JzEvMzAnLCdtaXJyb3InOidyYXdfYWNxdWlyZWRfbm9fdjgzJywnYWRhcHRpdmVfZXh0cmEnOmV4dHJhc30pCnRpbWluZz17J3N0YXR1cyc6J1BBU1MnLCdlbnRyeV91dGMnOkVOVFJZLCdjb21wbGV0ZWRfdXRjJzpkYXRldGltZS5ub3codGltZXpvbmUudXRjKS5pc29mb3JtYXQoKSwnc3R1ZGllcyc6bGVuKHVpZHMpLCdub3RlYm9va19lbnRyeV90b19zdWJtaXNzaW9uX3NlY29uZHMnOnRpbWUucGVyZl9jb3VudGVyKCktU1RBUlQsJ2Nhbm9uaWNhbF9sb29wX3NlY29uZHMnOmF1ZGl0Wydsb29wX3NlY29uZHMnXSwnY2Fub25pY2FsX2NvbnRyb2xsZXJfc2Vjb25kcyc6YXVkaXRbJ2NvbnRyb2xsZXJfc2Vjb25kcyddLCdyYXdfbG9vcF9zZWNvbmRzJzpyYXdfYXVkaXRbJ2xvb3Bfc2Vjb25kcyddLCdzY29wZSc6J0FjdHVhbCBLYWdnbGUgc2FtcGxlIGVudHJ5IHRocm91Z2hDU1YgYW5kUUE7IGluY2x1ZGVzIGJvb3RzdHJhcC93ZWlnaHRwYWNrL2NvcHkvcHJvY2Vzc3N0YXJ0dXA7IGV4Y2x1ZGVzIHByb3ZpZGVyIGFsbG9jYXRpb24vc2NvcmluZy4nfQpzYXZlKFdPUksvJ1JVTlRJTUVfVElNSU5HLmpzb24nLHRpbWluZykKc2F2ZShXT1JLLydJTkZFUkVOQ0VfUkVDRUlQVC5qc29uJyx7J3N0YXR1cyc6J1BBU1MnLCdtb2RlbHNfY291bnQnOjEwLCdjb21wb25lbnRzX2NvdW50Jzo2LCdzYW1wbGVfcGFyaXR5JzonUEFTUycgaWYgU0FNUExFX1JVTiBlbHNlICdOT1RfUFVCTElDX1NBTVBMRScsJ29mZmljaWFsX1VJRHMnOnVpZHMsJ3JlbGVhc2Vfc2hhMjU2JzpzaGEoRFMvJ1JFTEVBU0VfUEFDS0VULmpzb24nKSwnc3VibWlzc2lvbl9zaGEyNTYnOnNoYShXT1JLLydzdWJtaXNzaW9uLmNzdicpLCd0aW1pbmdfc2hhMjU2JzpzaGEoV09SSy8nUlVOVElNRV9USU1JTkcuanNvbicpLCdyZWFkX3JldXNlX3FhX3NoYTI1Nic6c2hhKFdPUksvJ1JFQURfUkVVU0VfUUEuanNvbicpLCdwYXJpdHlfcWFfc2hhMjU2JzpzaGEoV09SSy8nQ09NUE9ORU5UX1BBUklUWV9RQS5qc29uJyl9KQpmb3IgcCBpbiBSQVdfQkFOSy5nbG9iKCcqLnpwaycpOnAudW5saW5rKCkKc2lnbmFsLmFsYXJtKDApO3BoYXNlKCdSRVNJREVOVF9DT01QTEVURScsdGltaW5nPXRpbWluZyxyZWFkcz1yZWFkX3FhKQo='
src = base64.b64decode(SIXWAY_B64)
assert hashlib.sha256(src).hexdigest() == SIXWAY_SRC_SHA
script = Path('/tmp/sixway_stage.py'); script.write_bytes(src)
(WORK/'submission.csv').unlink(missing_ok=True)
t0 = time.perf_counter()
sixway_log = WORK/'sixway_stage.log'
with sixway_log.open('w', buffering=1) as log:
    proc = subprocess.run([sys.executable, '-u', str(script)], cwd=str(WORK), env=os.environ.copy(), stdout=log, stderr=subprocess.STDOUT)
SIXWAY_RC, SIXWAY_SECONDS = proc.returncode, time.perf_counter() - t0
shutil.rmtree('/tmp/rsna-sixway-resident-20261017', ignore_errors=True)  # Sixway is finished: free its packed weights and raw bank before the next stages
print(sixway_log.read_text()[-3000:], flush=True)
SIXWAY_CSV = WORK/'sixway_core.csv'
assert (WORK/'submission.csv').exists(), ('Sixway stage wrote no submission', SIXWAY_RC)
os.replace(WORK/'submission.csv', SIXWAY_CSV)  # from here on no submission.csv exists until the blend is written
base = pd.read_csv(SIXWAY_CSV, **RD)
assert valid_frame(base), 'Sixway output invalid'
uids = base.StudyInstanceUID.tolist()
SAMPLE_RUN = uids == PUBLIC_UIDS
SIXWAY_SHA = file_sha256(SIXWAY_CSV)
if SAMPLE_RUN:
    assert SIXWAY_RC == 0, ('Sixway child failed on the public sample', SIXWAY_RC)
    assert SIXWAY_SHA == SIXWAY_REF_SHA, SIXWAY_SHA
print('SIXWAY_STAGE', json.dumps(dict(rc=SIXWAY_RC, seconds=SIXWAY_SECONDS, sha256=SIXWAY_SHA, sample_run=SAMPLE_RUN, reference_match=SIXWAY_SHA == SIXWAY_REF_SHA, studies=len(uids))), flush=True)


In [ ]:
# Stage 2: H6 full-v4 Core6 + VM with weights = fixed SWA of EMA epochs 14-16 (dataset ilyakolb/rsna-h6-swa1416-weights-20261022); release code (Codex release f7d79463, dataset ilyakolb/rsna-h6-rank-full336-20261022 v1):
# infer_full.py byte for byte as a child process (full DICOM -> v4 views -> both H6 arms, FP32). Sixway is passed as its --k24 csv only to satisfy
# its interface; its own blend/submission stays in /tmp and is discarded. Only core6_raw_probabilities.csv and vm_raw_probabilities.csv are used.
import tarfile
H6_START = time.perf_counter()
ROOT = Path('/kaggle/input')
H6_SLUG = 'rsna-h6-rank-full336-20261022'
H6_RELEASE = 'f7d79463ab16720b1845c1ddbf07a890ecee10dcd7b08f1c6922c317195a525e'
H6_INFER_SHA = '781f18650de9e5ee62c1c76b928273a69b0df1cd8c358cd2a43218b808e9e720'
H6_REF = {"core6": {"1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191": [0.00016769830835983157, 0.0007910666754469275, 0.9937445521354675, 0.0006540754693560302, 0.17950047552585602, 0.0004346804053056985, 0.00445596082136035, 0.00802705716341734, 8.154210809152573e-05, 0.999182403087616, 0.0007777486462146044, 0.00021568583906628191], "1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776": [0.0008969111368060112, 0.0032502999529242516, 0.9396219849586487, 0.00017141684656962752, 0.9816479682922363, 0.9977694749832153, 0.9998636245727539, 0.19294922053813934, 0.5410638451576233, 0.0002504698059055954, 0.6783350110054016, 0.007058697286993265], "1.2.826.0.1.3680043.8.498.10067514707072572280263481548497591402": [4.3244250264251605e-05, 0.0006107890512794256, 0.000671436486300081, 0.0005449620657600462, 0.9950348138809204, 0.6851432919502258, 0.9935925602912903, 0.0003197762416675687, 0.0012797705130651593, 0.03116179257631302, 0.0013254709774628282, 5.4666739742970094e-05]}, "vm": {"1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191": [1.8759330487228e-05, 4.471530337468721e-05, 0.9492613077163696, 0.0002861027023755014, 0.01929854415357113, 0.0001343505718978122, 0.00014677502622362226, 0.00013546639820560813, 2.9533341148635373e-05, 0.9999672174453735, 0.0003936605644412339, 4.9900205340236425e-05], "1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776": [1.1697876288963016e-05, 2.6603378501022235e-05, 0.8056415915489197, 0.0010594003833830357, 0.6267951726913452, 0.9890310168266296, 0.999893069267273, 1.0339797881897539e-05, 0.14308016002178192, 3.736558574019e-05, 0.0017715945141389966, 0.00035496472264640033], "1.2.826.0.1.3680043.8.498.10067514707072572280263481548497591402": [9.871909242065158e-06, 0.00010627998562995344, 0.0006868021446280181, 0.001341736176982522, 0.9372856616973877, 0.00193175976164639, 0.715264618396759, 1.3678994037036318e-05, 0.00014097367238719016, 7.13561094016768e-05, 0.00030522668384946883, 8.003761467989534e-05]}}  # SWA EMA14-16, same recipe, public test3 (kernel rsna-h6-gold58-swa1416-20261022)
comp_hits = [p for p in [ROOT/'rsna-knee-abnormality-detection', ROOT/'competitions/rsna-knee-abnormality-detection'] if (p/'test_series').is_dir() and (p/'sample_submission.csv').is_file()]
assert len(comp_hits) >= 1, 'competition data not found'
COMP = comp_hits[0]
hits = [p for p in [ROOT/H6_SLUG, ROOT/'datasets/ilyakolb'/H6_SLUG] if (p/'RELEASE.json').exists() and file_sha256(p/'RELEASE.json') == H6_RELEASE]
assert len(hits) == 1, 'H6 release dataset not found'
h6mount = hits[0]; h6root = Path('/tmp/h6/release/full-v4'); shutil.rmtree(h6root, ignore_errors=True); h6root.mkdir(parents=True)  # rebuild_v4.py needs Path(__file__).parents[2]
if (h6mount/'payload.tar').exists():
    with tarfile.open(h6mount/'payload.tar') as t: t.extractall(h6root, filter='data')
else:
    shutil.copytree(h6mount/'payload', h6root, dirs_exist_ok=True)
assert file_sha256(h6root/'RELEASE.json') == H6_RELEASE
h6release = json.loads((h6root/'RELEASE.json').read_text())
# Kaggle unpacks nested archives on upload: wheels.tar may arrive as an extracted wheels/ folder. Every other release file is pinned by sha.
for name, digest in h6release['files'].items():
    if name == 'wheels.tar' and not (h6root/name).exists():
        continue
    assert file_sha256(h6root/name) == digest, name
assert h6release['files']['infer_full.py'] == H6_INFER_SHA
h6w = Path('/tmp/h6-profile-wheels'); shutil.rmtree(h6w, ignore_errors=True); h6w.mkdir()
if (h6root/'wheels.tar').exists():
    with tarfile.open(h6root/'wheels.tar') as t: t.extractall(h6w, filter='data')
else:
    for whl in list(h6root.rglob('*.whl')) + list(h6mount.rglob('*.whl')):
        if not (h6w/whl.name).exists(): shutil.copy(whl, h6w/whl.name)
from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename
compatible = set(sys_tags()); wh = [p for p in h6w.glob('*.whl') if parse_wheel_filename(p.name)[3] & compatible]
print('H6_WHEELS', sorted(p.name for p in wh), flush=True)
# infer_full.py pins every release file incl. wheels.tar; the only change: that one pin is skipped when Kaggle delivered wheels.tar unpacked.
PIN_OLD = "for n,h in release['files'].items():assert sha(HERE/n)==h,('EXACT_H6_RELEASE_PIN',n)"
PIN_NEW = "for n,h in release['files'].items():assert (n=='wheels.tar' and not (HERE/n).exists()) or sha(HERE/n)==h,('EXACT_H6_RELEASE_PIN',n)"
isrc = (h6root/'infer_full.py').read_text()
assert isrc.count(PIN_OLD) == 1
(h6root/'infer_full_kaggle.py').write_text(isrc.replace(PIN_OLD, PIN_NEW))
assert any(p.name.startswith('timm-1.0.29-') for p in wh)
h6deps = Path('/tmp/h6-pinned-deps'); shutil.rmtree(h6deps, ignore_errors=True); h6deps.mkdir()
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-index', '--no-deps', '--target', str(h6deps), *map(str, wh)], check=True)
h6env = os.environ.copy(); h6env.update(PYTHONPATH=str(h6deps), PYTHONNOUSERSITE='1', OMP_NUM_THREADS='2', MKL_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2')
# Pipelined H6 member (Claude, 22.10): same producer / models / FP32 / windows as infer_full.py, producers in 2 spawned workers
# while the parent runs both arms. A failed study gets NaN; > 3 % failures (or any on the public sample) -> no submission.
H6_FAST = {"h6_prep.py": ["IiIiMjIuMTAgKENsYXVkZSk6IEg2IGZ1bGwtdjQgcHJvZHVjZXIgZm9yIG9uZSBzdHVkeSwgcnVuIGluIGEgc3Bhd25lZCB3b3JrZXIgcHJvY2Vzcy4gVGhlIGJvZHkgaXMgaW5mZXJfZnVsbC5weSdzIHBlci1zdHVkeQpwcm9kdWNlciBibG9jayAocmVsZWFzZSBmN2Q3OTQ2MykgbGluZSBmb3IgbGluZTogcHJvZHVjZXIubmF0aXZlIC0+IG9yaWdpbmFsIGFuYXRvbXkgVk5ldCBvbiBjdWRhOjAgRlAzMiAtPiByZW5kZXJfb25fZGVtYW5kIHY0IC0+CndpbmRvd19yb3dzKGV2YWwsIHN0cmlkZSAxKSAtPiBjZW50cmUgY3JvcCAzNTIgKyBtZXRhIHByb2plY3Rpb24uIE1vZGVsIGluZmVyZW5jZSBzdGF5cyBpbiB0aGUgcGFyZW50IHByb2Nlc3MuIiIiCmltcG9ydCBvcwpvcy5lbnZpcm9uLnVwZGF0ZShPTVBfTlVNX1RIUkVBRFM9JzInLCBNS0xfTlVNX1RIUkVBRFM9JzInLCBPUEVOQkxBU19OVU1fVEhSRUFEUz0nMicpCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgZ2MsIHN5cywgdGVtcGZpbGUsIHRpbWUKaW1wb3J0IGltcG9ydGxpYiwgaW1wb3J0bGliLm1hY2hpbmVyeQojIFB5dGhvbiA+PSAzLjEyIHJlbW92ZWQgRmlsZUZpbmRlci5maW5kX21vZHVsZTsgdGhlIHZlbmRvcmVkIG1vbmFpIDAuNi4wIChhbmF0b215IFZOZXQpIHN0aWxsIGNhbGxzIGltcG9ydGVyLmZpbmRfbW9kdWxlKG5hbWUpLmxvYWRfbW9kdWxlKG5hbWUpCiMgcmlnaHQgYWZ0ZXIgaW1wb3J0X21vZHVsZShuYW1lKSBpbiBsb2FkX3N1Ym1vZHVsZXMuIFNoaW06IHJldHVybiBhIGxvYWRlciB3aG9zZSBsb2FkX21vZHVsZSBpcyBpbXBvcnRfbW9kdWxlIChtb2R1bGUgYWxyZWFkeSBpbXBvcnRlZCkuCmlmIG5vdCBoYXNhdHRyKGltcG9ydGxpYi5tYWNoaW5lcnkuRmlsZUZpbmRlciwgJ2ZpbmRfbW9kdWxlJyk6CiAgICBjbGFzcyBfTGVnYWN5TG9hZGVyOgogICAgICAgIGRlZiBfX2luaXRfXyhzZWxmLCBuYW1lKTogc2VsZi5uYW1lID0gbmFtZQogICAgICAgIGRlZiBsb2FkX21vZHVsZShzZWxmLCBuYW1lKTogcmV0dXJuIGltcG9ydGxpYi5pbXBvcnRfbW9kdWxlKG5hbWUpCiAgICBpbXBvcnRsaWIubWFjaGluZXJ5LkZpbGVGaW5kZXIuZmluZF9tb2R1bGUgPSBsYW1iZGEgc2VsZiwgbmFtZSwgcGF0aD1Ob25lOiBfTGVnYWN5TG9hZGVyKG5hbWUpCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKSEVSRSA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihIRVJFIC8gJ2g2JykpOyBzeXMucGF0aC5pbnNlcnQoMCwgc3RyKEhFUkUpKQpmcm9tIHNhbXBsaW5nIGltcG9ydCB3aW5kb3dfcm93cwppbXBvcnQgcmVidWlsZF92NCBhcyBwcm9kdWNlcgoKIyBQcm9maWxpbmcgd3JhcHBlcnMgKHNhbWUgY29tcHV0YXRpb25zLCBvbmx5IHRpbWVycyk6IERJQ09NIHJlYWQvZGVjb2RlLCBsYXRlcmFsaXR5IHJlc29sdmVyLCBvcmllbnRhdGlvbiAoaW5jbC4gcXVhbnRpemF0aW9uKS4KUFJPRiA9IHt9CmRlZiBfdGltZWQoa2V5LCBmbik6CiAgICBkZWYgd3JhcHBlcigqYXJncywgKiprd2FyZ3MpOgogICAgICAgIHQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgdHJ5OiByZXR1cm4gZm4oKmFyZ3MsICoqa3dhcmdzKQogICAgICAgIGZpbmFsbHk6IFBST0Zba2V5XSA9IFBST0YuZ2V0KGtleSwgMC4wKSArIHRpbWUucGVyZl9jb3VudGVyKCkgLSB0CiAgICByZXR1cm4gd3JhcHBlcgpjbGFzcyBfVGltZWRTb3VyY2VTdHVkeShwcm9kdWNlci5Tb3VyY2VTdHVkeSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgKmFyZ3MsICoqa3dhcmdzKToKICAgICAgICB0ID0gdGltZS5wZXJmX2NvdW50ZXIoKTsgc3VwZXIoKS5fX2luaXRfXygqYXJncywgKiprd2FyZ3MpOyBQUk9GWydyZWFkJ10gPSBQUk9GLmdldCgncmVhZCcsIDAuMCkgKyB0aW1lLnBlcmZfY291bnRlcigpIC0gdAogICAgZGVmIHNpZGVfaW5wdXRzKHNlbGYsIHJlc29sdmVyKToKICAgICAgICB0ID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIHRyeTogcmV0dXJuIHN1cGVyKCkuc2lkZV9pbnB1dHMocmVzb2x2ZXIpCiAgICAgICAgZmluYWxseTogUFJPRlsncmVzb2x2ZXInXSA9IFBST0YuZ2V0KCdyZXNvbHZlcicsIDAuMCkgKyB0aW1lLnBlcmZfY291bnRlcigpIC0gdApwcm9kdWNlci5Tb3VyY2VTdHVkeSA9IF9UaW1lZFNvdXJjZVN0dWR5CnByb2R1Y2VyLkMub3JpZW50ID0gX3RpbWVkKCdvcmllbnQnLCBwcm9kdWNlci5DLm9yaWVudCkKcHJvZHVjZXIuQy5xdWFudGl6ZV9zZXJpZXMgPSBfdGltZWQoJ3F1YW50aXplJywgcHJvZHVjZXIuQy5xdWFudGl6ZV9zZXJpZXMpCgpkZWYgaW5pdCgpOgogICAgdG9yY2guc2V0X251bV90aHJlYWRzKDIpOyB0b3JjaC5zZXRfbnVtX2ludGVyb3BfdGhyZWFkcygxKQogICAgdG9yY2guYmFja2VuZHMuY3VkYS5tYXRtdWwuYWxsb3dfdGYzMiA9IEZhbHNlOyB0b3JjaC5iYWNrZW5kcy5jdWRubi5hbGxvd190ZjMyID0gRmFsc2U7IHRvcmNoLmJhY2tlbmRzLmN1ZG5uLmJlbmNobWFyayA9IEZhbHNlCiAgICBkZXYgPSBvcy5lbnZpcm9uLmdldCgnSDZfUkVTT0xWRVJfREVWSUNFJywgJ2NwdScpICAjIG9yaWdpbmFsOiBMYXRlcmFsaXR5UmVzb2x2ZXIoT1JJRU5ULCBkZXZpY2U9J2NwdScpCiAgICBpZiBkZXYgIT0gJ2NwdSc6CiAgICAgICAgcHJvZHVjZXIuQy5yZXNvdXJjZXMoKQogICAgICAgIHByb2R1Y2VyLkMuUkVTT0xWRVIgPSBwcm9kdWNlci5DLkxhdGVyYWxpdHlSZXNvbHZlcihwcm9kdWNlci5DLk9SSUVOVCwgZGV2aWNlPWRldikKCmRlZiBwcmVwYXJlKHVpZCwgc3R1ZHlfcm93cywgc2VyaWVzX2RpciwgcWRpciwgdGFnPU5vbmUpOgogICAgdGFnID0gdGFnIG9yIHVpZAogICAgUFJPRi5jbGVhcigpOyB0aWNrID0gdGltZS5wZXJmX2NvdW50ZXIoKTsga25lZXMgPSBbXTsgdG0gPSB7J25hdGl2ZSc6IDAuMCwgJ2FuYXRvbXknOiAwLjAsICdyZW5kZXInOiAwLjAsICd3aW5kb3dzJzogMC4wLCAnc2F2ZSc6IDAuMH0KICAgIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KHByZWZpeD0naDYtdjQtJywgZGlyPScvdG1wJykgYXMgc2NyYXRjaDoKICAgICAgICB0ID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIHZhcmlhbnRzLCBkZWNpc2lvbiA9IHByb2R1Y2VyLm5hdGl2ZSh1aWQsIHN0dWR5X3Jvd3MsIFBhdGgoc2VyaWVzX2RpciksIFBhdGgoc2NyYXRjaCkgLyAnbmF0aXZlJykKICAgICAgICB0bVsnbmF0aXZlJ10gKz0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHQKICAgICAgICBmb3IgaywgKHN0dWR5LCBnZW9tZXRyeSwgcm91dGUpIGluIGVudW1lcmF0ZSh2YXJpYW50cyk6CiAgICAgICAgICAgIHQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgICAgIGRldiA9IG9zLmVudmlyb24uZ2V0KCdINl9BTkFUT01ZX0RFVklDRScsICdjdWRhOjAnKSAgIyBpbmZlcl9mdWxsOiBjdWRhOjA7IGFjY2VwdGVkIENQVSBjYW5hcmllcyAocmVidWlsZF92NC5tYWluKTogY3B1CiAgICAgICAgICAgIGFuYXRvbWljYWwsIF8gPSBwcm9kdWNlci5DLnJlc291cmNlcygpOyBhbmF0b21pY2FsLnRvKGRldikuZXZhbCgpCiAgICAgICAgICAgIGJhbmsgPSBwcm9kdWNlci5DLk1lbW9yeUFuY2hvckJhbmsoYW5hdG9taWNhbCwgZ2VvbWV0cnkpOyBiYW5rLmRldiA9IGRldgogICAgICAgICAgICB3aXRoIHRvcmNoLmluZmVyZW5jZV9tb2RlKCk6IGFuY2hvcnMgPSBiYW5rLnJlY29yZChzdHVkeVsnU3R1ZHlJbnN0YW5jZVVJRCddLCBzdHVkeSkKICAgICAgICAgICAgaWYgYW5jaG9yc1snam9pbnRfbHBzJ10gaXMgTm9uZTogcmFpc2UgVmFsdWVFcnJvcignTk9fQU5BVE9NSUNBTF9PUl9HRU9NRVRSWV9BTkNIT1InKQogICAgICAgICAgICB0bVsnYW5hdG9teSddICs9IHRpbWUucGVyZl9jb3VudGVyKCkgLSB0OyB0ID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgICAgICByZW5kZXJlZCA9IHByb2R1Y2VyLlIucmVuZGVyX29uX2RlbWFuZChzdHVkeSwgYW5jaG9ycywgeydjb21wb25lbnRzX3Jvb3QnOiBzdHIocHJvZHVjZXIuQ09NUE9ORU5UUyksICdpbWFnZV9zaXplJzogMzg0LCAndmlld3MnOiAndjQnLCAnbG9jYWxfbW0nOiAxMDAuLCAnaW50ZW5zaXR5X3BvbGljeSc6ICdzZXJpZXNfY29tcGFuZGVkJ30pCiAgICAgICAgICAgIHB4LCBtZXRhID0gcmVuZGVyZWRbJ3B4J10sIHJlbmRlcmVkWydtZXRhJ107IHB4aGFzaCA9IHByb2R1Y2VyLmFycmF5X3NoYShweCk7IG1ldGFoYXNoID0gcHJvZHVjZXIuYXJyYXlfc2hhKG1ldGEpCiAgICAgICAgICAgIHRtWydyZW5kZXInXSArPSB0aW1lLnBlcmZfY291bnRlcigpIC0gdDsgdCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICAgICAgYmxvY2tzID0gW3sqKmIsICd2aWV3cyc6IFsoMCwgdiwgc3RhcnQpIGZvciB2LCBzdGFydCBpbiBiWyd2aWV3cyddXX0gZm9yIGIgaW4gcmVuZGVyZWRbJ2Jsb2NrcyddXQogICAgICAgICAgICByZWZzID0gbnAuYXNhcnJheSh3aW5kb3dfcm93cyhibG9ja3MsIGxhbWJkYSBmaWxlLCByb3c6IG1ldGFbcm93XSwgJ2V2YWwnLCAzLCBldmFsX3N0cmlkZT0xLCBjb3Zlcl9zdHJpZGU9MywgcGh5c19tbT0zLjUsIHJuZz1Ob25lLCBzbGljZV9qaXR0ZXI9LjI1LCB0cmFpbl9zYW1wbGluZz0nY292ZXInKSwgbnAuaW50NjQpCiAgICAgICAgICAgIGFzc2VydCBsZW4ocmVmcykgPiAwCiAgICAgICAgICAgIGltYWdlcyA9IG5wLmFzY29udGlndW91c2FycmF5KHB4W3JlZnNbOiwgMjpdXVsuLi4sIDE2OjM2OCwgMTY6MzY4XSk7IHByb2plY3RlZCA9IG1ldGFbcmVmc1s6LCAxXV1bOiwgWzEsIDMsIDQsIDUsIDYsIDcsIDAsIDIsIDEwXV0uY29weSgpCiAgICAgICAgICAgIGFzc2VydCBpbWFnZXMuc2hhcGUgPT0gKGxlbihyZWZzKSwgMywgMzUyLCAzNTIpIGFuZCBwcm9qZWN0ZWQuc2hhcGUgPT0gKGxlbihyZWZzKSwgOSkKICAgICAgICAgICAgdG1bJ3dpbmRvd3MnXSArPSB0aW1lLnBlcmZfY291bnRlcigpIC0gdDsgdCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICAgICAgaXAgPSBQYXRoKHFkaXIpIC8gZid7dGFnfV97a31faW1hZ2VzLm5weSc7IG1wID0gUGF0aChxZGlyKSAvIGYne3RhZ31fe2t9X21ldGEubnB5JwogICAgICAgICAgICBucC5zYXZlKGlwLCBpbWFnZXMpOyBucC5zYXZlKG1wLCBwcm9qZWN0ZWQpCiAgICAgICAgICAgIHRtWydzYXZlJ10gKz0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHQKICAgICAgICAgICAga25lZXMuYXBwZW5kKHsnaW1hZ2VzJzogc3RyKGlwKSwgJ21ldGEnOiBzdHIobXApLCAncm91dGUnOiByb3V0ZSwgJ3BpeGVsX3Jvd3MnOiBsZW4ocHgpLCAnd2luZG93cyc6IGxlbihyZWZzKSwgJ3B4X3NoYTI1Nic6IHB4aGFzaCwgJ21ldGFfc2hhMjU2JzogbWV0YWhhc2gsICdhbmF0b215X21ldGhvZCc6IGFuY2hvcnNbJ21ldGhvZCddLCAnYW5hdG9teV9kZXZpY2UnOiBkZXZ9KQogICAgICAgICAgICBkZWwgaW1hZ2VzLCBwcm9qZWN0ZWQsIHB4LCBtZXRhLCByZW5kZXJlZCwgcmVmczsgZ2MuY29sbGVjdCgpCiAgICAgICAgYXNzZXJ0IGxlbih2YXJpYW50cykgPT0gKDIgaWYgZGVjaXNpb25bJ2JpbGF0ZXJhbCddIGVsc2UgMSkKICAgIHJldHVybiB7J2lkJzogdWlkLCAnYmlsYXRlcmFsJzogYm9vbChkZWNpc2lvblsnYmlsYXRlcmFsJ10pLCAndmFyaWFudHMnOiBsZW4odmFyaWFudHMpLCAna25lZXMnOiBrbmVlcywgJ3ByZXBfc2Vjb25kcyc6IHRpbWUucGVyZl9jb3VudGVyKCkgLSB0aWNrLCAnc3RhZ2Vfc2Vjb25kcyc6IHsqKnRtLCAqKnsnbmF0aXZlXycgKyBrOiB2IGZvciBrLCB2IGluIFBST0YuaXRlbXMoKX19LCAncGlkJzogb3MuZ2V0cGlkKCl9Cg==", "746c9ab116733f65c531d2d76ee55fec5c8f87356cf145712f9c1d613ec9b082"], "h6_fast.py": ["IiIiMjIuMTAgKENsYXVkZSk6IHBpcGVsaW5lZCBINiBmdWxsLXY0IEVNQTEwIG1lbWJlciAocmVsZWFzZSBmN2Q3OTQ2MykuIFNhbWUgcHJvZHVjZXIsIG1vZGVscywgRlAzMiBtYXRoIGFuZCB3aW5kb3dzIGFzIGluZmVyX2Z1bGwucHk7Cm9ubHkgdGhlIHNjaGVkdWxlIGNoYW5nZXM6IE4gc3Bhd25lZCBwcm9kdWNlciB3b3JrZXJzIHByZXBhcmUgdXBjb21pbmcgc3R1ZGllcyB3aGlsZSB0aGUgcGFyZW50IHJ1bnMgYm90aCBhcm1zIChjdWRhOjAgY29yZTYsIGN1ZGE6MSB2bSkuCkEgc3R1ZHkgd2hvc2UgcHJvZHVjZXIgb3IgbW9kZWwgZmFpbHMgZ2V0cyBOYU4gKHJlY29yZGVkKTsgdGhlIGNhbGxlciBkZWNpZGVzIHRoZSBmYWlsdXJlIHBvbGljeS4gV3JpdGVzIGNvcmU2L3ZtIHJhdyBwcm9iYWJpbGl0aWVzLiIiIgppbXBvcnQgb3MKb3MuZW52aXJvbi51cGRhdGUoT01QX05VTV9USFJFQURTPScyJywgTUtMX05VTV9USFJFQURTPScyJywgT1BFTkJMQVNfTlVNX1RIUkVBRFM9JzInKQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IGFyZ3BhcnNlLCBjc3YsIGdjLCBoYXNobGliLCBqc29uLCBtdWx0aXByb2Nlc3NpbmcgYXMgbXAsIHN5cywgdGhyZWFkaW5nLCB0aW1lLCB0cmFjZWJhY2sKZnJvbSBjb25jdXJyZW50LmZ1dHVyZXMgaW1wb3J0IFByb2Nlc3NQb29sRXhlY3V0b3IsIFRocmVhZFBvb2xFeGVjdXRvcgpmcm9tIGNvbmN1cnJlbnQuZnV0dXJlcy5wcm9jZXNzIGltcG9ydCBCcm9rZW5Qcm9jZXNzUG9vbAppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCkhFUkUgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50CnN5cy5wYXRoLmluc2VydCgwLCBzdHIoSEVSRSAvICdoNicpKTsgc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihIRVJFKSkKZnJvbSBtb2RlbCBpbXBvcnQgTW9kZWwKZnJvbSBwdWJsaWNfbWF0aCBpbXBvcnQgTmV0CmltcG9ydCBoNl9wcmVwCmRlZiBzaGEocCk6CiAgICB3aXRoIHAub3BlbigncmInKSBhcyBmOiByZXR1cm4gaGFzaGxpYi5maWxlX2RpZ2VzdChmLCAnc2hhMjU2JykuaGV4ZGlnZXN0KCkKZGVmIHJlYWQocCk6IHJldHVybiBqc29uLmxvYWRzKHAucmVhZF90ZXh0KCkpCmRlZiBzYXZlKHAsIHgpOgogICAgdCA9IHAud2l0aF9zdWZmaXgoJy5wYXJ0aWFsJyk7IHQud3JpdGVfdGV4dChqc29uLmR1bXBzKHgsIGluZGVudD0yKSArICdcbicpOyBvcy5yZXBsYWNlKHQsIHApCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCk7IGFwLmFkZF9hcmd1bWVudCgnLS1kYXRhJywgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKTsgYXAuYWRkX2FyZ3VtZW50KCctLW91dCcsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFwLmFkZF9hcmd1bWVudCgnLS13b3JrZXJzJywgdHlwZT1pbnQsIGRlZmF1bHQ9Mik7IGFwLmFkZF9hcmd1bWVudCgnLS1kZWFkbGluZS11bml4JywgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjApOyBhcC5hZGRfYXJndW1lbnQoJy0tbWF4LXN0dWRpZXMnLCB0eXBlPWludCwgZGVmYXVsdD0wKTsgYXAuYWRkX2FyZ3VtZW50KCctLXJlcGVhdCcsIHR5cGU9aW50LCBkZWZhdWx0PTEpOyBhcC5hZGRfYXJndW1lbnQoJy0tcHJlY2lzaW9uJywgY2hvaWNlcz1bJ2ZwMzInLCAnZnAxNiddLCBkZWZhdWx0PSdmcDMyJyk7IGFwLmFkZF9hcmd1bWVudCgnLS1wcmVmZXRjaC1haGVhZCcsIHR5cGU9aW50LCBkZWZhdWx0PTApOyBhcC5hZGRfYXJndW1lbnQoJy0tcHJlZmV0Y2gtdGhyZWFkcycsIHR5cGU9aW50LCBkZWZhdWx0PTE2KTsgYXAuYWRkX2FyZ3VtZW50KCctLXdlaWdodHMtbWFuaWZlc3QnLCB0eXBlPVBhdGgsIGRlZmF1bHQ9Tm9uZSkKICAgIGEgPSBhcC5wYXJzZV9hcmdzKCk7IGEub3V0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHJlbGVhc2UgPSByZWFkKEhFUkUgLyAnUkVMRUFTRS5qc29uJyk7IGFzc2VydCByZWxlYXNlWydINl9lcG9jaCddID09IDEwICAjIHJlbGVhc2UgY29kZS9wcm9kdWNlcjsgd2VpZ2h0cyBtYXkgYmUgb3ZlcnJpZGRlbiBiZWxvdwogICAgYXNzZXJ0IHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgYW5kIHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCkgPj0gMgogICAgdG9yY2guc2V0X251bV90aHJlYWRzKDIpOyB0b3JjaC5zZXRfbnVtX2ludGVyb3BfdGhyZWFkcygxKQogICAgdG9yY2guY3VkYS5zZXRfcGVyX3Byb2Nlc3NfbWVtb3J5X2ZyYWN0aW9uKC43NSwgMCk7IHRvcmNoLmN1ZGEuc2V0X3Blcl9wcm9jZXNzX21lbW9yeV9mcmFjdGlvbiguNzUsIDEpCiAgICB0b3JjaC5iYWNrZW5kcy5jdWRhLm1hdG11bC5hbGxvd190ZjMyID0gRmFsc2U7IHRvcmNoLmJhY2tlbmRzLmN1ZG5uLmFsbG93X3RmMzIgPSBGYWxzZTsgdG9yY2guYmFja2VuZHMuY3Vkbm4uYmVuY2htYXJrID0gRmFsc2UKICAgIHdpdGggKGEuZGF0YSAvICdzYW1wbGVfc3VibWlzc2lvbi5jc3YnKS5vcGVuKCkgYXMgZjogciA9IGNzdi5EaWN0UmVhZGVyKGYpOyBjb2x1bW5zID0gci5maWVsZG5hbWVzOyBzYW1wbGUgPSBsaXN0KHIpCiAgICBuYW1lcyA9IHJlbGVhc2VbJ3RhcmdldHMnXTsgaWRzID0gW3hbJ1N0dWR5SW5zdGFuY2VVSUQnXSBmb3IgeCBpbiBzYW1wbGVdCiAgICBhc3NlcnQgY29sdW1ucyA9PSBbJ1N0dWR5SW5zdGFuY2VVSUQnLCAqbmFtZXNdIGFuZCBpZHMgYW5kIGxlbihpZHMpID09IGxlbihzZXQoaWRzKSkKICAgIGlmIGEubWF4X3N0dWRpZXM6IGlkcyA9IGlkc1s6YS5tYXhfc3R1ZGllc10KICAgIGlmIGEucmVwZWF0ID4gMTogaWRzID0gaWRzICogYS5yZXBlYXQgICMgdGltaW5nIGJlbmNobWFyayBvbmx5OiB0aGUgc2FtZSBzdHVkaWVzIGFnYWluLCBvdXRwdXRzIGtleWVkIGJ5IHBvc2l0aW9uCiAgICByb3dzID0gbGlzdChjc3YuRGljdFJlYWRlcigoYS5kYXRhIC8gJ3Rlc3Rfc2VyaWVzLmNzdicpLm9wZW4oKSkpCiAgICBieSA9IHt9CiAgICBmb3IgeCBpbiByb3dzOiBieS5zZXRkZWZhdWx0KHhbJ1N0dWR5SW5zdGFuY2VVSUQnXSwgW10pLmFwcGVuZCh4KQogICAgYXNzZXJ0IHNldChpZHMpIDw9IHNldChieSkKICAgIHRfc3RhcnQgPSB0aW1lLnBlcmZfY291bnRlcigpOyBtb2RlbHMgPSB7fQogICAgIyBXZWlnaHRzOiB0aGUgcmVsZWFzZSBFTUExMCBieSBkZWZhdWx0OyAtLXdlaWdodHMtbWFuaWZlc3Qge2FybToge3BhdGgsIHNoYTI1NiwgZXBvY2hzfX0gc3dhcHMgaW4gb3RoZXIgY2hlY2twb2ludHMgb2YgdGhlIHNhbWUgcGFja2FnZQogICAgIyAoZS5nLiBmaXhlZCBTV0Egb2YgRU1BIGVwb2NocyAxNC0xNikuIFNhbWUgYXJjaGl0ZWN0dXJlLCBzdHJpY3QgbG9hZCwgc2hhIHBpbm5lZC4KICAgIHdtYW4gPSBqc29uLmxvYWRzKGEud2VpZ2h0c19tYW5pZmVzdC5yZWFkX3RleHQoKSkgaWYgYS53ZWlnaHRzX21hbmlmZXN0IGVsc2UgTm9uZQogICAgd2VpZ2h0c191c2VkID0ge30KICAgIGZvciBhcm0sIGRldmljZSBpbiBbKCdjb3JlNicsICdjdWRhOjAnKSwgKCd2bScsICdjdWRhOjEnKV06CiAgICAgICAgaWYgd21hbjoKICAgICAgICAgICAgZmlsZSA9IFBhdGgod21hblthcm1dWydwYXRoJ10pOyBhc3NlcnQgc2hhKGZpbGUpID09IHdtYW5bYXJtXVsnc2hhMjU2J10sICgnV0VJR0hUU19TSEEnLCBhcm0pCiAgICAgICAgICAgIGNwID0gdG9yY2gubG9hZChmaWxlLCBtYXBfbG9jYXRpb249J2NwdScsIG1tYXA9VHJ1ZSwgd2VpZ2h0c19vbmx5PUZhbHNlKQogICAgICAgICAgICBhc3NlcnQgY3BbJ2FybSddID09IGFybSBhbmQgbGlzdChjcFsnZXBvY2hzJ10pID09IGxpc3Qod21hblthcm1dWydlcG9jaHMnXSkgYW5kIGNwWydhdmVyYWdpbmdfc291cmNlJ10gPT0gJ0VNQScgYW5kIGNwWydwYWNrYWdlX3NoYTI1NiddID09IHJlbGVhc2VbJ3NvdXJjZV9wYWNrYWdlX3NoYTI1NiddCiAgICAgICAgZWxzZToKICAgICAgICAgICAgaW5mbyA9IHJlbGVhc2VbJ3dlaWdodHMnXVthcm1dOyBmaWxlID0gSEVSRSAvIGluZm9bJ2ZpbGUnXTsgYXNzZXJ0IHNoYShmaWxlKSA9PSBpbmZvWydzaGEyNTYnXQogICAgICAgICAgICBjcCA9IHRvcmNoLmxvYWQoZmlsZSwgbWFwX2xvY2F0aW9uPSdjcHUnLCBtbWFwPVRydWUsIHdlaWdodHNfb25seT1GYWxzZSkKICAgICAgICAgICAgYXNzZXJ0IGNwWydhcm0nXSA9PSBhcm0gYW5kIGNwWydlcG9jaCddID09IDEwIGFuZCBjcFsna2luZCddID09ICdlbWEnIGFuZCBjcFsncGFja2FnZV9zaGEyNTYnXSA9PSByZWxlYXNlWydzb3VyY2VfcGFja2FnZV9zaGEyNTYnXQogICAgICAgIHdlaWdodHNfdXNlZFthcm1dID0geydmaWxlJzogc3RyKGZpbGUpLCAnc2hhMjU2Jzogc2hhKGZpbGUpfQogICAgICAgIHB1YmxpYyA9IE5ldCgnY29udm5leHRfc21hbGwuZmJfaW4yMmtfZnRfaW4xa18zODQnLCBOb25lLCBoaWRkZW49Mzg0LCBkcm9wPS4xLCBjcm9wPTM1Miwgc3BhdGlhbD1UcnVlLCBzeW5fZGVjb21wPVRydWUsIGZtX2F1eD0xMSkKICAgICAgICBtb2RlbCA9IE1vZGVsKE5vbmUsIGFybSwgOCwgcHVibGljPXB1YmxpYyk7IG1vZGVsLmxvYWRfc3RhdGVfZGljdChjcFsnbW9kZWwnXSwgc3RyaWN0PVRydWUpOyBkZWwgY3AsIHB1YmxpYzsgZ2MuY29sbGVjdCgpOyBtb2RlbC50byhkZXZpY2UpLmV2YWwoKTsgbW9kZWxzW2FybV0gPSAobW9kZWwsIGRldmljZSkKICAgIGxvYWRfc2Vjb25kcyA9IHRpbWUucGVyZl9jb3VudGVyKCkgLSB0X3N0YXJ0CiAgICBjYW5hcmllcyA9IHJlbGVhc2VbJ3B1YmxpY190ZXN0M19jYWNoZV9jYW5hcmllcyddCiAgICBmcDMyX3JlcnVucyA9IFtdCiAgICBkZWYgcHJlZGljdChhcm0sIGltYWdlcywgbWV0YSk6CiAgICAgICAgbW9kZWwsIGRldmljZSA9IG1vZGVsc1thcm1dOyB0b3JjaC5jdWRhLnNldF9kZXZpY2UoZGV2aWNlKQogICAgICAgIHdpdGggdG9yY2guaW5mZXJlbmNlX21vZGUoKToKICAgICAgICAgICAgeCA9IHRvcmNoLmZyb21fbnVtcHkoaW1hZ2VzKVtOb25lXS50byhkZXZpY2UpOyBtID0gdG9yY2guZnJvbV9udW1weShtZXRhKVtOb25lXS50byhkZXZpY2UpOyBtYXNrID0gdG9yY2gub25lcygoMSwgbGVuKGltYWdlcykpLCBkdHlwZT10b3JjaC5ib29sLCBkZXZpY2U9ZGV2aWNlKQogICAgICAgICAgICBpZiBhLnByZWNpc2lvbiA9PSAnZnAxNic6ICAjIGF1dG9jYXN0IGZwMTYgb24gVDQgdGVuc29yIGNvcmVzOyBhbnkgbm9uLWZpbml0ZSBvdXRwdXQgLT4gZXhhY3QgRlAzMiByZXJ1biBvZiB0aGF0IGtuZWUKICAgICAgICAgICAgICAgIHdpdGggdG9yY2guYXV0b2Nhc3QoJ2N1ZGEnLCBkdHlwZT10b3JjaC5mbG9hdDE2KToKICAgICAgICAgICAgICAgICAgICB6ID0gbW9kZWwuZ29sZCh4LCBtLCBtYXNrKQogICAgICAgICAgICAgICAgcHJvYiA9IHouZmxvYXQoKS5zaWdtb2lkKClbMF0uY3B1KCkubnVtcHkoKQogICAgICAgICAgICAgICAgaWYgbm90IChucC5pc2Zpbml0ZShwcm9iKS5hbGwoKSBhbmQgKChwcm9iID49IDApICYgKHByb2IgPD0gMSkpLmFsbCgpKToKICAgICAgICAgICAgICAgICAgICBmcDMyX3JlcnVucy5hcHBlbmQoYXJtKTsgcHJvYiA9IG1vZGVsLmdvbGQoeCwgbSwgbWFzaykuc2lnbW9pZCgpWzBdLmZsb2F0KCkuY3B1KCkubnVtcHkoKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgcHJvYiA9IG1vZGVsLmdvbGQoeCwgbSwgbWFzaykuc2lnbW9pZCgpWzBdLmZsb2F0KCkuY3B1KCkubnVtcHkoKQogICAgICAgICAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKGRldmljZSkKICAgICAgICBhc3NlcnQgcHJvYi5zaGFwZSA9PSAoMTIsKSBhbmQgbnAuaXNmaW5pdGUocHJvYikuYWxsKCkgYW5kICgocHJvYiA+PSAwKSAmIChwcm9iIDw9IDEpKS5hbGwoKQogICAgICAgIHJldHVybiBwcm9iCiAgICBxZGlyID0gUGF0aCgnL3RtcC9oNnEnKTsgcWRpci5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgY3R4ID0gbXAuZ2V0X2NvbnRleHQoJ3NwYXduJykKICAgIGRlZiBuZXdfcG9vbCgpOiByZXR1cm4gUHJvY2Vzc1Bvb2xFeGVjdXRvcihtYXhfd29ya2Vycz1hLndvcmtlcnMsIG1wX2NvbnRleHQ9Y3R4LCBpbml0aWFsaXplcj1oNl9wcmVwLmluaXQpCiAgICBwb29sID0gbmV3X3Bvb2woKTsgcGVuZGluZyA9IHt9OyBueHQgPSAwOyBkZXB0aCA9IGEud29ya2VycyArIDEKICAgIGRlZiBmaWxsKCk6CiAgICAgICAgbm9ubG9jYWwgbnh0CiAgICAgICAgd2hpbGUgbnh0IDwgbGVuKGlkcykgYW5kIGxlbihwZW5kaW5nKSA8IGRlcHRoOgogICAgICAgICAgICB1ID0gaWRzW254dF07IHBlbmRpbmdbbnh0XSA9IHBvb2wuc3VibWl0KGg2X3ByZXAucHJlcGFyZSwgdSwgYnlbdV0sIHN0cihhLmRhdGEgLyAndGVzdF9zZXJpZXMnKSwgc3RyKHFkaXIpLCBmJ3F7bnh0fScpOyBueHQgKz0gMQogICAgIyBESUNPTSBwcmVmZXRjaCAocmVhZHMgb25seSwgYnl0ZXMgZGlzY2FyZGVkKTogd2FybXMgdGhlIHBhZ2UgY2FjaGUgb2YgL2thZ2dsZS9pbnB1dCBmb3IgdGhlIG5leHQgc3R1ZGllcyB3aXRoIHBhcmFsbGVsIEkvTywKICAgICMgc28gdGhlIHByb2R1Y2VyJ3Mgc2VxdWVudGlhbCBweWRpY29tIHJlYWRzIGhpdCBtZW1vcnkuIERvZXMgbm90IGNoYW5nZSBhbnkgY29tcHV0YXRpb24uCiAgICBjb25zdW1lciA9IHsnaSc6IDAsICdzdG9wJzogRmFsc2V9OyBwZl9zdGF0cyA9IHsnc3R1ZGllcyc6IDAsICdieXRlcyc6IDAsICdzZWNvbmRzJzogMC4wfQogICAgZGVmIF9yZWFkKHBhdGgpOgogICAgICAgIG4gPSAwCiAgICAgICAgdHJ5OgogICAgICAgICAgICB3aXRoIG9wZW4ocGF0aCwgJ3JiJywgYnVmZmVyaW5nPTApIGFzIGZoOgogICAgICAgICAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgICAgICAgICBiID0gZmgucmVhZCg4IDw8IDIwKQogICAgICAgICAgICAgICAgICAgIGlmIG5vdCBiOiBicmVhawogICAgICAgICAgICAgICAgICAgIG4gKz0gbGVuKGIpCiAgICAgICAgZXhjZXB0IE9TRXJyb3I6CiAgICAgICAgICAgIHBhc3MKICAgICAgICByZXR1cm4gbgogICAgZGVmIHByZWZldGNoX2xvb3AoKToKICAgICAgICB3aXRoIFRocmVhZFBvb2xFeGVjdXRvcihtYXhfd29ya2Vycz1hLnByZWZldGNoX3RocmVhZHMpIGFzIGlvOgogICAgICAgICAgICBmb3IgaiwgdSBpbiBlbnVtZXJhdGUoaWRzKToKICAgICAgICAgICAgICAgIHdoaWxlIG5vdCBjb25zdW1lclsnc3RvcCddIGFuZCBqID4gY29uc3VtZXJbJ2knXSArIGEucHJlZmV0Y2hfYWhlYWQ6IHRpbWUuc2xlZXAoMC4yKQogICAgICAgICAgICAgICAgaWYgY29uc3VtZXJbJ3N0b3AnXTogcmV0dXJuCiAgICAgICAgICAgICAgICB0MCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICAgICAgICAgIGZpbGVzID0gW29zLnBhdGguam9pbihkLCBmKSBmb3IgZCwgXywgZnMgaW4gb3Mud2FsayhhLmRhdGEgLyAndGVzdF9zZXJpZXMnIC8gdSkgZm9yIGYgaW4gZnNdCiAgICAgICAgICAgICAgICBwZl9zdGF0c1snYnl0ZXMnXSArPSBzdW0oaW8ubWFwKF9yZWFkLCBmaWxlcykpOyBwZl9zdGF0c1snc3R1ZGllcyddICs9IDE7IHBmX3N0YXRzWydzZWNvbmRzJ10gKz0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHQwCiAgICBpZiBhLnByZWZldGNoX2FoZWFkID4gMDoKICAgICAgICB0aHJlYWRpbmcuVGhyZWFkKHRhcmdldD1wcmVmZXRjaF9sb29wLCBkYWVtb249VHJ1ZSkuc3RhcnQoKQogICAgb3V0ID0geydjb3JlNic6IG5wLmZ1bGwoKGxlbihpZHMpLCAxMiksIG5wLm5hbiksICd2bSc6IG5wLmZ1bGwoKGxlbihpZHMpLCAxMiksIG5wLm5hbil9CiAgICByZWNvcmRzID0gW107IGZhaWx1cmVzID0gW107IGdwdV9zZWNvbmRzID0gMC4wOyByZXN0YXJ0cyA9IDAKICAgIHdpdGggVGhyZWFkUG9vbEV4ZWN1dG9yKG1heF93b3JrZXJzPTIpIGFzIHRocmVhZHM6CiAgICAgICAgZmlsbCgpCiAgICAgICAgZm9yIGksIHVpZCBpbiBlbnVtZXJhdGUoaWRzKToKICAgICAgICAgICAgY29uc3VtZXJbJ2knXSA9IGk7IHRpY2sgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgICAgIGlmIGEuZGVhZGxpbmVfdW5peCBhbmQgdGltZS50aW1lKCkgPiBhLmRlYWRsaW5lX3VuaXg6CiAgICAgICAgICAgICAgICBmYWlsdXJlcy5hcHBlbmQoeydpZCc6IHVpZCwgJ2Vycm9yJzogJ0RFQURMSU5FJ30pOyBwZW5kaW5nLnBvcChpLCBOb25lKTsgY29udGludWUKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIHJlcyA9IHBlbmRpbmcucG9wKGkpLnJlc3VsdCgpCiAgICAgICAgICAgICAgICBleGNlcHQgQnJva2VuUHJvY2Vzc1Bvb2w6CiAgICAgICAgICAgICAgICAgICAgcmVzdGFydHMgKz0gMTsgcG9vbC5zaHV0ZG93bih3YWl0PUZhbHNlLCBjYW5jZWxfZnV0dXJlcz1UcnVlKTsgcG9vbCA9IG5ld19wb29sKCkKICAgICAgICAgICAgICAgICAgICBmb3IgaiBpbiBsaXN0KHBlbmRpbmcpOiBwZW5kaW5nW2pdID0gcG9vbC5zdWJtaXQoaDZfcHJlcC5wcmVwYXJlLCBpZHNbal0sIGJ5W2lkc1tqXV0sIHN0cihhLmRhdGEgLyAndGVzdF9zZXJpZXMnKSwgc3RyKHFkaXIpLCBmJ3F7an0nKQogICAgICAgICAgICAgICAgICAgIHJlcyA9IHBvb2wuc3VibWl0KGg2X3ByZXAucHJlcGFyZSwgdWlkLCBieVt1aWRdLCBzdHIoYS5kYXRhIC8gJ3Rlc3Rfc2VyaWVzJyksIHN0cihxZGlyKSwgZidxe2l9JykucmVzdWx0KCkKICAgICAgICAgICAgICAgIGZpbGwoKQogICAgICAgICAgICAgICAgZzAgPSB0aW1lLnBlcmZfY291bnRlcigpOyBwcm9icyA9IHsnY29yZTYnOiBbXSwgJ3ZtJzogW119OyBjYW5hcnkgPSBOb25lCiAgICAgICAgICAgICAgICBmb3Iga24gaW4gcmVzWydrbmVlcyddOgogICAgICAgICAgICAgICAgICAgIGlmIHVpZCBpbiBjYW5hcmllczogICMgcmVjb3JkZWQsIG5vdCBhc3NlcnRlZDogdGhlIGNhbGxlciBjb21wYXJlcyBwcmVkaWN0aW9ucyB3aXRoIHRoZSBhY2NlcHRlZC1jYWNoZSByZWZlcmVuY2UKICAgICAgICAgICAgICAgICAgICAgICAgY2FuYXJ5ID0gZGljdCh2YXJpYW50cz1yZXNbJ3ZhcmlhbnRzJ10sIHB4X2V4YWN0PWtuWydweF9zaGEyNTYnXSA9PSBjYW5hcmllc1t1aWRdWydwaXhlbF9hcnJheV9zaGEyNTYnXSwgbWV0YV9leGFjdD1rblsnbWV0YV9zaGEyNTYnXSA9PSBjYW5hcmllc1t1aWRdWydtZXRhX2FycmF5X3NoYTI1NiddKQogICAgICAgICAgICAgICAgICAgIGltYWdlcyA9IG5wLmxvYWQoa25bJ2ltYWdlcyddKTsgbWV0YSA9IG5wLmxvYWQoa25bJ21ldGEnXSkKICAgICAgICAgICAgICAgICAgICBmdXRzID0ge2FybTogdGhyZWFkcy5zdWJtaXQocHJlZGljdCwgYXJtLCBpbWFnZXMsIG1ldGEpIGZvciBhcm0gaW4gWydjb3JlNicsICd2bSddfQogICAgICAgICAgICAgICAgICAgIGZvciBhcm0gaW4gZnV0czogcHJvYnNbYXJtXS5hcHBlbmQoZnV0c1thcm1dLnJlc3VsdCgpKQogICAgICAgICAgICAgICAgICAgIGRlbCBpbWFnZXMsIG1ldGEKICAgICAgICAgICAgICAgIGZvciBhcm0gaW4gcHJvYnM6IG91dFthcm1dW2ldID0gbnAubWF4KG5wLnN0YWNrKHByb2JzW2FybV0pLCBheGlzPTApCiAgICAgICAgICAgICAgICBncHVfc2Vjb25kcyArPSB0aW1lLnBlcmZfY291bnRlcigpIC0gZzAKICAgICAgICAgICAgICAgIHJlY29yZHMuYXBwZW5kKHsnaWQnOiB1aWQsICdpbmRleCc6IGksICd0X2RvbmUnOiB0aW1lLnBlcmZfY291bnRlcigpIC0gdF9zdGFydCwgJ2NhbmFyeSc6IGNhbmFyeSwgJ2JpbGF0ZXJhbCc6IHJlc1snYmlsYXRlcmFsJ10sICdwcmVwX3NlY29uZHMnOiByZXNbJ3ByZXBfc2Vjb25kcyddLCAnc3RhZ2Vfc2Vjb25kcyc6IHJlcy5nZXQoJ3N0YWdlX3NlY29uZHMnKSwgJ2dwdV9zZWNvbmRzJzogdGltZS5wZXJmX2NvdW50ZXIoKSAtIGcwLCAnd2FpdF9zZWNvbmRzJzogZzAgLSB0aWNrLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICdrbmVlcyc6IFt7azogdiBmb3IgaywgdiBpbiBrbi5pdGVtcygpIGlmIGsgbm90IGluICgnaW1hZ2VzJywgJ21ldGEnKX0gZm9yIGtuIGluIHJlc1sna25lZXMnXV19KQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4OgogICAgICAgICAgICAgICAgZmlsbCgpCiAgICAgICAgICAgICAgICBmYWlsdXJlcy5hcHBlbmQoeydpZCc6IHVpZCwgJ2Vycm9yJzogcmVwcihleClbOjUwMF0sICd0cmFjZSc6IHRyYWNlYmFjay5mb3JtYXRfZXhjKClbLTE1MDA6XX0pCiAgICAgICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgICAgICBmb3IgZiBpbiBxZGlyLmdsb2IoZidxe2l9XyonKTogZi51bmxpbmsobWlzc2luZ19vaz1UcnVlKQogICAgICAgICAgICBpZiAoaSArIDEpICUgMjUgPT0gMCBvciBpICsgMSA9PSBsZW4oaWRzKToKICAgICAgICAgICAgICAgIHNhdmUoYS5vdXQgLyAnSDZfRkFTVF9QUk9HUkVTUy5qc29uJywgeydjaGVja2VkJzogaSArIDEsICd0b3RhbCc6IGxlbihpZHMpLCAnZmFpbHVyZXMnOiBsZW4oZmFpbHVyZXMpLCAnZWxhcHNlZCc6IHRpbWUucGVyZl9jb3VudGVyKCkgLSB0X3N0YXJ0LCAnYXRfdW5peCc6IHRpbWUudGltZSgpfSkKICAgICAgICAgICAgcHJpbnQoJ0g2X1NUVURZX0RPTkUnLCBpICsgMSwgbGVuKGlkcyksIHVpZCwgcm91bmQodGltZS5wZXJmX2NvdW50ZXIoKSAtIHRpY2ssIDMpLCAnZmFpbCcgaWYgZmFpbHVyZXMgYW5kIGZhaWx1cmVzWy0xXVsnaWQnXSA9PSB1aWQgZWxzZSAnb2snLCBmbHVzaD1UcnVlKQogICAgY29uc3VtZXJbJ3N0b3AnXSA9IFRydWU7IHBvb2wuc2h1dGRvd24od2FpdD1GYWxzZSwgY2FuY2VsX2Z1dHVyZXM9VHJ1ZSkKICAgIGZvciBhcm0sIHAgaW4gb3V0Lml0ZW1zKCk6CiAgICAgICAgd2l0aCAoYS5vdXQgLyAoYXJtICsgJ19yYXdfcHJvYmFiaWxpdGllcy5jc3YnKSkub3BlbigndycsIG5ld2xpbmU9JycpIGFzIGY6CiAgICAgICAgICAgIHcgPSBjc3Yud3JpdGVyKGYpOyB3LndyaXRlcm93KGNvbHVtbnMpOyB3LndyaXRlcm93cyhbdWlkLCAqcFtpXV0gZm9yIGksIHVpZCBpbiBlbnVtZXJhdGUoaWRzKSkKICAgIGNhbmFyeV9wYXNzID0gW3JbJ2lkJ10gZm9yIHIgaW4gcmVjb3JkcyBpZiByLmdldCgnY2FuYXJ5JykgYW5kIHJbJ2NhbmFyeSddWydweF9leGFjdCddIGFuZCByWydjYW5hcnknXVsnbWV0YV9leGFjdCddIGFuZCByWydjYW5hcnknXVsndmFyaWFudHMnXSA9PSAxXQogICAgYXVkaXQgPSB7J3N0YXR1cyc6ICdET05FJywgJ3N0dWRpZXMnOiBsZW4oaWRzKSwgJ2ZhaWx1cmVzJzogZmFpbHVyZXMsICduX2ZhaWx1cmVzJzogbGVuKGZhaWx1cmVzKSwgJ3dvcmtlcnMnOiBhLndvcmtlcnMsICdwb29sX3Jlc3RhcnRzJzogcmVzdGFydHMsCiAgICAgICAgICAgICAnbW9kZWxfbG9hZF9zZWNvbmRzJzogbG9hZF9zZWNvbmRzLCAndG90YWxfc2Vjb25kcyc6IHRpbWUucGVyZl9jb3VudGVyKCkgLSB0X3N0YXJ0LCAnZ3B1X3NlY29uZHMnOiBncHVfc2Vjb25kcywgJ3JlY29yZHMnOiByZWNvcmRzLAogICAgICAgICAgICAgJ3B1YmxpY190ZXN0M19jYW5hcmllc19wYXNzZWQnOiBjYW5hcnlfcGFzcywgJ3JlbGVhc2Vfc2hhMjU2Jzogc2hhKEhFUkUgLyAnUkVMRUFTRS5qc29uJyksICdwcmVjaXNpb24nOiAnRlAzMl9ub1RGMzJfbm9hdXRvY2FzdCcgaWYgYS5wcmVjaXNpb24gPT0gJ2ZwMzInIGVsc2UgJ0ZQMTZfYXV0b2Nhc3RfZnAzMl9mYWxsYmFjaycsICdmcDMyX3JlcnVucyc6IGxlbihmcDMyX3JlcnVucyksICd3ZWlnaHRzJzogd2VpZ2h0c191c2VkLCAncHJlZmV0Y2gnOiBkaWN0KHBmX3N0YXRzLCBhaGVhZD1hLnByZWZldGNoX2FoZWFkLCB0aHJlYWRzPWEucHJlZmV0Y2hfdGhyZWFkcyksICdhdF91bml4JzogdGltZS50aW1lKCl9CiAgICBzYXZlKGEub3V0IC8gJ0g2X0ZBU1RfQVVESVQuanNvbicsIGF1ZGl0KTsgcHJpbnQoJ0g2X0ZBU1RfRE9ORScsIGxlbihpZHMpLCBsZW4oZmFpbHVyZXMpLCByb3VuZChhdWRpdFsndG90YWxfc2Vjb25kcyddLCAxKSwgZmx1c2g9VHJ1ZSkKaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzoKICAgIG1haW4oKQo=", "1497a0eff784b32f9979fe98791682966ce339042957ad52ddc6d64f23c1d318"]}
for fname, (b64, digest) in H6_FAST.items():
    data = base64.b64decode(b64); assert hashlib.sha256(data).hexdigest() == digest, fname
    (h6root/fname).write_bytes(data)
_w = [p for p in [ROOT/'rsna-h6-swa1416-weights-20261022', ROOT/'datasets/ilyakolb/rsna-h6-swa1416-weights-20261022'] if (p/'MANIFEST.json').is_file()]
assert len(_w) == 1, 'SWA14-16 weights dataset not found'
_wm = json.loads((_w[0]/'MANIFEST.json').read_text())
assert _wm['core6']['sha256'] == 'c9d916ffe853af4dcdee3452178ce62c6d558c15b86c58881b6956a4b146b015' and _wm['vm']['sha256'] == '9e0960154610daf25a63a4d3ede3737bf69d06079d649f11a72b54d85390f7d6'
WMAN = Path('/tmp/h6_weights_manifest.json')
WMAN.write_text(json.dumps({arm: dict(path=str(_w[0]/_wm[arm]['file']), sha256=_wm[arm]['sha256'], epochs=_wm[arm]['epochs']) for arm in ['core6', 'vm']}))
H6_RECIPE = {"workers": 3, "precision": "fp16", "env": {"H6_RESOLVER_DEVICE": "cuda:0"}, "args": ["--prefetch-ahead", "8", "--prefetch-threads", "16"], "parity_tol": 0.005, "source": "Gold58 on Kaggle T4x2 22.10 (cold DICOM): 6.1 s/study steady (FP32 exact 14.9), 0 failures; fp16 arms max|dp| 0.0024 mean 7e-5 vs accepted-cache FP32, Gold AUC unchanged (core6 0.9315, vm 0.937); resolver on cuda identical predictions; prefetch 16 threads x 8 studies"}
H6_WORKERS = int(H6_RECIPE['workers'])
H6_PARITY_TOL = float(H6_RECIPE['parity_tol'])  # vs accepted-cache predictions of the same checkpoints on the 3 public studies
H6OUT = Path('/tmp/h6_out'); shutil.rmtree(H6OUT, ignore_errors=True)
h6_budget = 9 * 3600 - SIXWAY_SECONDS - (time.perf_counter() - H6_START) - 1500  # 25 min reserve for start-up, blend and write
import datetime as _dt
H6_WALL_STOP = _dt.datetime(2026, 10, 22, 23, 45, tzinfo=_dt.timezone.utc).timestamp()  # = 1792712700, 2026-10-22 23:45 UTC = 23.10 02:45 MSK (Ilya) (competition closes 23:59 UTC)
H6_DEADLINE = min(time.time() + max(1200, h6_budget), H6_WALL_STOP)
h6_log = WORK/'h6_stage.log'
t0 = time.perf_counter()
with h6_log.open('w', buffering=1) as log:
    try:
        hproc = subprocess.run([sys.executable, '-u', str(h6root/'h6_fast.py'), '--data', str(COMP), '--out', str(H6OUT), '--workers', str(H6_WORKERS), '--deadline-unix', str(H6_DEADLINE), '--precision', H6_RECIPE['precision'], '--weights-manifest', str(WMAN), *H6_RECIPE.get('args', [])],
                               env=dict(h6env, **H6_RECIPE['env']), stdout=log, stderr=subprocess.STDOUT, timeout=max(1500, int(h6_budget) + 600))
        H6_RC = hproc.returncode
    except subprocess.TimeoutExpired:
        H6_RC = 'timeout'
H6_SECONDS = time.perf_counter() - t0
print(h6_log.read_text()[-3000:], flush=True)
for name in ['H6_FAST_AUDIT.json', 'H6_FAST_PROGRESS.json']:
    if (H6OUT/name).exists():
        shutil.copy(H6OUT/name, WORK/('h6_' + name))
assert not (WORK/'submission.csv').exists()
assert (H6OUT/'H6_FAST_AUDIT.json').exists(), ('H6 stage wrote no audit', H6_RC)
h6_audit = json.loads((H6OUT/'H6_FAST_AUDIT.json').read_text())
fail_ids = sorted(x['id'] for x in h6_audit['failures'])
h6 = {}
for arm in ['core6', 'vm']:
    p = H6OUT/(arm + '_raw_probabilities.csv')
    assert p.exists(), ('H6 stage wrote no raw probabilities', arm, H6_RC)
    f = pd.read_csv(p, **RD)
    assert list(f.columns) == ['StudyInstanceUID', *TARGETS] and sorted(f.StudyInstanceUID) == sorted(uids), ('H6 UIDs differ from Sixway', arm)
    f = f.set_index('StudyInstanceUID').loc[uids].reset_index()
    v = f[TARGETS].to_numpy(float); nan_rows = np.isnan(v).any(axis=1)
    assert sorted(f.StudyInstanceUID[nan_rows]) == fail_ids and bool(((v[~nan_rows] >= 0) & (v[~nan_rows] <= 1)).all()), ('H6 values invalid', arm)
    f.to_csv(WORK/('h6_' + arm + '_raw.csv'), index=False)
    h6[arm] = f
H6_MAX_FAIL = max(1, int(0.03 * len(uids)))
h6_parity = dict(weights={arm: _wm[arm]['sha256'] for arm in ['core6', 'vm']}, recipe=H6_RECIPE, rc=H6_RC, seconds=H6_SECONDS, failures=len(fail_ids), max_failures=H6_MAX_FAIL, workers=H6_WORKERS,
                 gpu_seconds=h6_audit['gpu_seconds'], total_seconds=h6_audit['total_seconds'], first_failures=h6_audit['failures'][:3])
deadline_ids = sorted(x['id'] for x in h6_audit['failures'] if x.get('error') == 'DEADLINE')
h6_parity['deadline_failures'] = len(deadline_ids); h6_parity['wall_stop_utc'] = H6_WALL_STOP; h6_parity['hard_failures'] = len(fail_ids) - len(deadline_ids)
assert h6_parity['hard_failures'] <= H6_MAX_FAIL, ('too many H6 failures (not deadline)', h6_parity)  # DEADLINE studies -> base rank
if SAMPLE_RUN:
    assert H6_RC == 0 and not fail_ids, ('H6 child failed on the public sample', h6_parity)
    h6_parity['canary_exact'] = sorted(h6_audit['public_test3_canaries_passed']) == sorted(uids)
    h6_parity['canary_detail'] = {r['id'][-6:]: dict(r['canary'], windows=[k['windows'] for k in r['knees']], anatomy=[k['anatomy_method'] for k in r['knees']], prep=r['prep_seconds']) for r in h6_audit['records']}
    for arm in ['core6', 'vm']:
        ref = np.asarray([H6_REF[arm][u] for u in uids], float)
        d = np.abs(ref - h6[arm][TARGETS].to_numpy(float))
        h6_parity[arm + '_max_abs_difference'] = float(d.max()); h6_parity[arm + '_mean_abs_difference'] = float(d.mean())
    print('H6_PARITY', json.dumps(h6_parity), flush=True)
    for arm in ['core6', 'vm']:
        assert h6_parity[arm + '_max_abs_difference'] <= H6_PARITY_TOL, h6_parity
print('H6_STAGE_COMPLETE', json.dumps(h6_parity), flush=True)


In [ ]:
# Stage 3: per-target weighted mean of percentile ranks (Ilya 22.10). C6 = H6 Core6 SWA EMA14-16, VM = H6 VM SWA EMA14-16, S6 = Sixway.
# rank = per-column pct rank over the scored studies (pandas, method average). Columns with S6 only stay Sixway byte for byte. A failed H6 study takes the Sixway rank.
W = {'ACL': {'S6': 0.75, 'C6': 0.25}, 'MCL': {'S6': 0.75, 'C6': 0.25}, 'Medial Meniscus': {'S6': 0.75, 'C6': 0.25}, 'Lateral Meniscus': {'S6': 0.75, 'C6': 0.25}, 'Medial OA': {'S6': 0.5, 'C6': 0.25, 'VM': 0.25}, 'Lateral OA': {'S6': 0.5, 'C6': 0.25, 'VM': 0.25}, 'PF OA': {'S6': 0.5, 'C6': 0.25, 'VM': 0.25}, 'Effusion': {'S6': 1.0}, 'Synovitis': {'S6': 1.0}, "Baker's": {'S6': 0.5, 'C6': 0.25, 'VM': 0.25}, 'Contusion': {'S6': 0.5, 'C6': 0.25, 'VM': 0.25}, 'Fracture': {'S6': 1.0}}
assert list(W) == TARGETS and all(abs(sum(w.values()) - 1) < 1e-12 for w in W.values())
def pct_rank(frame):
    return frame[TARGETS].rank(axis=0, method='average', pct=True).to_numpy(float)
RS = pct_rank(base)
RC = pct_rank(h6['core6']); RC = np.where(np.isnan(RC), RS, RC)
RV = pct_rank(h6['vm']); RV = np.where(np.isnan(RV), RS, RV)
SRC = {'C6': RC, 'VM': RV, 'S6': RS}
final = base[TARGETS].to_numpy(float).copy()
for j, t in enumerate(TARGETS):
    if W[t] == {'S6': 1.0}:
        continue
    final[:, j] = sum(wt * SRC[s][:, j] for s, wt in W[t].items())
assert np.isfinite(final).all() and ((final >= 0) & (final <= 1)).all()
s6_only = [j for j, t in enumerate(TARGETS) if W[t] == {'S6': 1.0}]
assert np.array_equal(final[:, s6_only], base[TARGETS].to_numpy(float)[:, s6_only])
out = pd.DataFrame(final, columns=TARGETS)
out.insert(0, 'StudyInstanceUID', uids)
out.to_csv(WORK/'submission.csv', index=False)
check = pd.read_csv(WORK/'submission.csv', **RD)
assert check.StudyInstanceUID.tolist() == uids and float(np.abs(check[TARGETS].to_numpy(float) - final).max()) <= 1e-15
receipt = dict(status='PASS', ladder='FINAL_SIX_H6SWA_ENS_RANK', studies=len(uids), weights=W, blend_space='percentile_rank', s6_only_columns_sixway_raw=True,
               sixway_rc=SIXWAY_RC, sixway_seconds=SIXWAY_SECONDS, sixway_sha256=SIXWAY_SHA, sixway_reference_match=SIXWAY_SHA == SIXWAY_REF_SHA, sixway_sample_run=SAMPLE_RUN,
               h6_release=H6_RELEASE, h6=h6_parity, submission_sha256=file_sha256(WORK/'submission.csv'))
(WORK/'LADDER_RECEIPT.json').write_text(json.dumps(receipt, indent=2) + '\n')
print('LADDER_COMPLETE', json.dumps(receipt), flush=True)
